# Product-Layer Valuation Engine Tests -- Fixed Accrued, the Three Atomic Index Cashflows, and Interest Rate Stream

Exercises, against a single shared three-component USD curve:

- `ValuationEngineProductFixedAccrued` -- pure fixed-coupon cashflow, no index.
- The three atomic index-linked cashflow engines built on the anchored-index analytics layer:
  `ValuationEngineProductOvernightIndexCompositeCashflow` (SOFR, daily-compounded),
  `ValuationEngineProductIBORIndexCashflow` (LIBOR-3M, single native-tenor period), and
  `ValuationEngineProductIBORCompoundingCashflow` (LIBOR-3M, 3 native-tenor periods
  geometrically compounded, both `SPREAD_EXCLUSIVE_COMPOUND` and `FLAT_COMPOUND`).
- `ValuationEngineProductInterestRateStream` -- a full leg (portfolio of the above), fixed and
  floating.

For every engine: PV at several `value_date`s (fully forward, on the payment date with
realized fixings, fully matured), `pv01()` against a closed form, and `get_risk()` against an
independent finite-difference (parallel-bump) estimate, checked block-by-block against the
curve's own component structure.

**Gotcha carried over from the other test notebooks in this directory:**
`model.discount_factor(index, date)` defaults to `calc_grad=False`, which flips
`requires_grad` to `False` *in place* on the curve component's shared state-data tensor -- a
plain (non-`calc_grad`) lookup run after an engine has already built its differentiable graph
on the same model would silently zero out that engine's later `.backward()`/`get_gradient()`.
Every closed-form comparison below passes `calc_grad=True` for that reason, even when the
result is only used for a print/assert.

Also covers the three `ARTIFICIAL PRODUCT` engines at the bottom of `valuation_engine.py`
(sections 12-14): `ValuationEngineProductGenericForward` (implied forward rate off an arbitrary
index), `ValuationEngineProductGenericSpread` (spread between two data-convention-driven
instruments -- currently blocked from construction, see section 13), and
`ValuationEngineProductGenericForwardSpread` (spread between two implied forward rates; two
further suspected library bugs documented there, not patched).


In [1]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import numpy as np
import pandas as pd
import torch

from fixedincomelib import *
from fixedincomelib.yield_curve.valuation_engine import (
    ValuationEngineProductFixedAccrued,
    ValuationEngineProductOvernightIndexCompositeCashflow,
    ValuationEngineProductIBORIndexCashflow,
    ValuationEngineProductIBORCompoundingCashflow,
    ValuationEngineProductInterestRateStream,
    _to_float,
)
from fixedincomelib.product.linear_products import (
    ProductFixedAccrued,
    ProductOvernightIndexCompositeCashflow,
    ProductIBORIndexCashflow,
    ProductIBORCompoundingCashflow,
    ProductInterestRateStream,
)
from fixedincomelib.valuation.valuation_parameters import (
    ValuationParametersCollection,
    AnalyticValParam,
    FundingIndexParameter,
)

print("Fixed Income Library is loaded.")


Fixed Income Library is loaded.


## Build a shared USD curve

Three components: `SOFR-1B` (overnight projection, upward-sloping IFR), `USD-LIBOR-BBA-3M`
(IBOR projection, `REFERENCE`d off `SOFR-1B`), and `SOFR-1B-FLAT` (the discounting/funding
curve, also `REFERENCE`d off `SOFR-1B`, solved `BRENT` to a flat zero spread -- i.e. it tracks
`SOFR-1B` one-for-one, the way OIS discounting works in practice). Every engine below discounts
off `SOFR-1B-FLAT` and projects off either `SOFR-1B` or `USD-LIBOR-BBA-3M`.

`build_model(value_date, ...)` rebuilds the whole model at a given value date (and optionally
bumped IFR levels, for the finite-difference risk checks) -- there is no in-place "reprice as of
a different date" operation in this codebase, so every value-date comparison below constructs
its own model.


In [2]:
TENORS = ['3M', '6M', '1Y', '2Y', '5Y', '10Y', '30Y']
FLAT_TENORS = ['1Y', '10Y', '30Y']

SOFR_IFR = [0.0430, 0.0432, 0.0435, 0.0440, 0.0445, 0.0448, 0.0450]
LIBOR_IFR = [0.0450, 0.0452, 0.0455, 0.0458, 0.0460, 0.0462, 0.0465]
FLAT_IFR = [0.0, 0.0, 0.0]

bm_list = [
    qfCreateBuildMethod('YC_OVERNIGHT_INDEX_ELEMENT', {
        'TARGET': 'SOFR-1B',
        'INSTANTANEOUS FORWARD RATE': 'USD-SOFR-OIS-1B-IFR',
    }),
    qfCreateBuildMethod('YC_IBOR_ELEMENT', {
        'TARGET': 'USD-LIBOR-BBA-3M',
        'REFERENCE': 'SOFR-1B',
        'INSTANTANEOUS FORWARD RATE': 'USD-LIBOR-BBA-3M-IFR',
    }),
    qfCreateBuildMethod('YC_FUNDING_ELEMENT', {
        'TARGET': 'SOFR-1B-FLAT',
        'REFERENCE': 'SOFR-1B',
        'INSTANTANEOUS FORWARD RATE': 'USD-SOFR-OIS-1B-FLAT-IFR',
    }),
    qfCreateBuildMethod('YC_COMMON', {
        'TARGET': 'USD',
        'FUNDING PARAMETERS': 'SOFR-1B-FLAT',
        'SOLVER METHOD': 'BRENT',
    }),
]
build_method_collection = qfCreateModelBuildMethodCollection(bm_list)


def _mkdf(index, values):
    df = pd.DataFrame(index=index)
    df['values'] = values
    return df


def build_data(sofr_ifr=None, libor_ifr=None, flat_ifr=None):
    return qfCreateDataCollection([
        qfCreateData1D('INSTANTANEOUS FORWARD RATE', 'USD-SOFR-OIS-1B-IFR',
                        _mkdf(TENORS, sofr_ifr if sofr_ifr is not None else SOFR_IFR)),
        qfCreateData1D('INSTANTANEOUS FORWARD RATE', 'USD-LIBOR-BBA-3M-IFR',
                        _mkdf(TENORS, libor_ifr if libor_ifr is not None else LIBOR_IFR)),
        qfCreateData1D('INSTANTANEOUS FORWARD RATE', 'USD-SOFR-OIS-1B-FLAT-IFR',
                        _mkdf(FLAT_TENORS, flat_ifr if flat_ifr is not None else FLAT_IFR)),
    ])


def build_model(value_date, sofr_bump=0.0, libor_bump=0.0, flat_bump=0.0):
    dc = build_data(
        sofr_ifr=[v + sofr_bump for v in SOFR_IFR],
        libor_ifr=[v + libor_bump for v in LIBOR_IFR],
        flat_ifr=[v + flat_bump for v in FLAT_IFR],
    )
    return qfCreateModel(value_date, 'YIELD_CURVE', dc, build_method_collection)


VALUE_DATE = '2026-07-17'
yc = build_model(VALUE_DATE)

sofr_index = IndexRegistry().get('SOFR-1B')
sofr_composite = IndexRegistry().get('USD-SOFR-COMPOUND')
libor_3m = IndexRegistry().get('USD-LIBOR-BBA-3M')
funding_identifier = FundingIdentifierRegistry().get('SOFR-1B-FLAT')

vpc = ValuationParametersCollection([
    FundingIndexParameter({
        'FUNDING INDEX': 'SOFR-1B-FLAT',
        'CURRENCIES': '',
        'FUNDING INDICES': '',
        'UNDERLYING FUNDING INDEX': '',
    }),
])

print('component_order_    :', yc.component_order_)

# gradient_lengths_ is populated lazily by get_gradient() (harvested off each component's own
# .grad, defaulting to zero-length if never called) -- force it once here with a no-op
# (reset=False, nothing has a live graph yet) read so N_STATE/SLICES below are correct.
yc.get_gradient(reset=False)
print('gradient_lengths_    :', yc.gradient_lengths_)
N_STATE = sum(yc.gradient_lengths_)
print('N_STATE              :', N_STATE)


component_order_    : ['SOFR-1B', 'USD-LIBOR-BBA-3M', 'SOFR-1B-FLAT']
gradient_lengths_    : [7, 7, 3]
N_STATE              : 17


`get_risk`'s gradient vector is ordered per `component_order_` above -- the cells below slice
it into per-component blocks by that same order rather than assuming a fixed layout, and compare
each block only against a finite difference that bumps *that* component's own target (since
`SOFR-1B-FLAT` and `USD-LIBOR-BBA-3M` are both `REFERENCE`d off `SOFR-1B` and solved/calibrated
independently, a bump to one target does not move the others' own state).

In [3]:
def block_slices(yc_model):
    slices, offset = {}, 0
    for name, length in zip(yc_model.component_order_, yc_model.gradient_lengths_):
        slices[name] = slice(offset, offset + length)
        offset += length
    return slices


SLICES = block_slices(yc)
print(SLICES)


{'SOFR-1B': slice(0, 7, None), 'USD-LIBOR-BBA-3M': slice(7, 14, None), 'SOFR-1B-FLAT': slice(14, 17, None)}


## 1. `ValuationEngineProductFixedAccrued`

A single fixed-coupon cashflow, 3M, 4.5% coupon, `ACT/360`, no index -- its only curve
dependency is the `SOFR-1B-FLAT` discounting leg.

In [4]:
EFFECTIVE = Date('2026-08-19')
COUPON = 0.045
NOTIONAL = 10_000_000.0

fa_biz_conv = ql.Following
fa_hol_conv = ql.UnitedStates(ql.UnitedStates.GovernmentBond)
fa_termination = add_period(EFFECTIVE, Period('3M'), fa_biz_conv, fa_hol_conv)

fa_product = ProductFixedAccrued(
    EFFECTIVE, TermOrDate(fa_termination), PayOrReceive.RECEIVE,
    Currency('USD'), NOTIONAL, COUPON, AccrualBasis.new('ACTUAL/360'),
    business_day_convention=fa_biz_conv, holiday_convention=fa_hol_conv,
)
print('effective  :', fa_product.effective_date)
print('termination:', fa_product.termination_date)
print('payment    :', fa_product.payment_date)
print('accrued    :', fa_product.accrued)


effective  : August 19th, 2026
termination: November 19th, 2026
payment    : November 19th, 2026
accrued    : 0.25555555555555554


### 1a. PV vs. closed form, at a fully-forward value date

In [5]:
fa_engine = ValuationEngineProductFixedAccrued(yc, vpc, fa_product, ValuationRequest.PV)
fa_engine.calculate_value()

tau = fa_product.accrued
df_pay = yc.discount_factor(funding_identifier, fa_product.payment_date, calc_grad=True)
expected_pv = NOTIONAL * tau * COUPON * df_pay

print('engine PV  :', _to_float(fa_engine.value))
print('closed PV  :', float(expected_pv.detach()))
assert abs(_to_float(fa_engine.value) - float(expected_pv.detach())) < 1e-6
assert fa_engine.cash == 0.0
print('PASSED')


engine PV  : 113316.99022703551
closed PV  : 113316.99022703551
PASSED


### 1b. `pv01()` exact closed form (PV is affine in the coupon)

In [6]:
pv01 = fa_engine.pv01()
expected_pv01 = NOTIONAL * tau * float(df_pay.detach()) * 1e-4
print('engine pv01 :', pv01, ' expected:', expected_pv01)
assert abs(pv01 - expected_pv01) < 1e-8
print('PASSED')


engine pv01 : 251.8155338378567  expected: 251.8155338378567
PASSED


### 1c. Settlement-date branches: on payment date (cash realized), and fully matured

In [7]:
yc_on_pay = build_model(fa_product.payment_date)
fa_on_pay = ValuationEngineProductFixedAccrued(yc_on_pay, vpc, fa_product, ValuationRequest.PV)
fa_on_pay.calculate_value()
print('value_date == payment_date -> value:', fa_on_pay.value, ' cash:', fa_on_pay.cash, ' df:', fa_on_pay.df_)
assert fa_on_pay.df_ == 1.0 and fa_on_pay.value == fa_on_pay.cash and fa_on_pay.cash != 0.0

matured_date = add_period(fa_product.payment_date, Period('1D'), fa_biz_conv, fa_hol_conv)
yc_matured = build_model(matured_date)
fa_matured = ValuationEngineProductFixedAccrued(yc_matured, vpc, fa_product, ValuationRequest.PV)
fa_matured.calculate_value()
print('value_date >  payment_date -> value:', fa_matured.value, ' cash:', fa_matured.cash)
assert fa_matured.value == 0.0 and fa_matured.cash == 0.0
print('PASSED')


value_date == payment_date -> value: 115000.0  cash: 115000.0  df: 1.0
value_date >  payment_date -> value: 0.0  cash: 0.0
PASSED


### 1d. `get_risk()` vs. finite difference

`USD-LIBOR-BBA-3M` never enters a fixed cashflow at all, so that block must be exactly zero.
`SOFR-1B-FLAT` is `REFERENCE`d off `SOFR-1B` (its discount factor is `DF(SOFR-1B) *
DF(own zero-spread state)`), so discounting through `SOFR-1B-FLAT` alone still leaves a real,
nonzero analytic gradient on the `SOFR-1B` block too -- **not** a bug, and not something to
assert away as zero (a wrong assumption caught by this very check on the first pass writing
this notebook). Both nonzero blocks are checked against their own independent finite
difference, bumping one component's target at a time (see the shared-setup note above for why
that matters on a `REFERENCE`d + `BRENT`-solved curve).

In [8]:
EPS = 1e-6
REL_TOL = 1e-4

grad = np.zeros(N_STATE)
fa_engine.get_risk(gradient=grad)
print('SOFR-1B block     sum:', grad[SLICES['SOFR-1B']].sum())
print('USD-LIBOR-BBA-3M   sum:', grad[SLICES['USD-LIBOR-BBA-3M']].sum())
print('SOFR-1B-FLAT block sum:', grad[SLICES['SOFR-1B-FLAT']].sum())

assert abs(grad[SLICES['USD-LIBOR-BBA-3M']].sum()) < 1e-8

base_pv = _to_float(fa_engine.value)


def bumped_pv(engine_cls, product, **bumps):
    yc_bumped = build_model(VALUE_DATE, **bumps)
    eng = engine_cls(yc_bumped, vpc, product, ValuationRequest.PV)
    eng.calculate_value()
    return _to_float(eng.value)


fd_sofr = (bumped_pv(ValuationEngineProductFixedAccrued, fa_product, sofr_bump=EPS) - base_pv) / EPS
fd_flat = (bumped_pv(ValuationEngineProductFixedAccrued, fa_product, flat_bump=EPS) - base_pv) / EPS
print('finite-diff (bump SOFR-1B)     :', fd_sofr, ' vs analytic:', grad[SLICES['SOFR-1B']].sum())
print('finite-diff (bump SOFR-1B-FLAT):', fd_flat, ' vs analytic:', grad[SLICES['SOFR-1B-FLAT']].sum())
assert abs(fd_sofr - grad[SLICES['SOFR-1B']].sum()) / abs(fd_sofr) < REL_TOL
assert abs(fd_flat - grad[SLICES['SOFR-1B-FLAT']].sum()) / abs(fd_flat) < REL_TOL
print('PASSED')


SOFR-1B block     sum: -38807.18843391627
USD-LIBOR-BBA-3M   sum: 0.0
SOFR-1B-FLAT block sum: -38807.18843391627
finite-diff (bump SOFR-1B)     : -38807.18178697862  vs analytic: -38807.18843391627
finite-diff (bump SOFR-1B-FLAT): -38807.18178697862  vs analytic: -38807.18843391627
PASSED


## 2. Atomic index cashflow 1/3 -- `ValuationEngineProductOvernightIndexCompositeCashflow`

SOFR compounded daily over a 3M period (`USD-SOFR-COMPOUND`, already registered in
`static_files/indices.yaml`), 15bp spread, `leverage=1.5`.

In [9]:
on_biz_conv = sofr_index.payment_business_day_conv
on_hol_conv = sofr_index.settlement_holiday
on_termination = add_period(EFFECTIVE, Period('3M'), on_biz_conv, on_hol_conv)
ON_SPREAD = 0.0015
ON_LEVERAGE = 1.5

on_product = ProductOvernightIndexCompositeCashflow(
    EFFECTIVE, TermOrDate(on_termination), PayOrReceive.RECEIVE, sofr_composite,
    ON_SPREAD, Currency('USD'), NOTIONAL, ON_LEVERAGE,
    payment_business_day_convention=on_biz_conv, payment_holiday_convention=on_hol_conv,
)
print('leverage:', on_product.leverage, ' spread:', on_product.spread)


leverage: 1.5  spread: 0.0015


### 2a. PV vs. closed form (fully forward)

In [10]:
on_engine = ValuationEngineProductOvernightIndexCompositeCashflow(yc, vpc, on_product, ValuationRequest.PV)
on_engine.calculate_value()

on_tau = on_product.accrued
df_eff = yc.discount_factor(sofr_index, on_product.effective_date, calc_grad=True)
df_term = yc.discount_factor(sofr_index, on_product.termination_date, calc_grad=True)
on_expected_forward = (df_eff / df_term - 1.0) / on_tau
on_df_pay = yc.discount_factor(funding_identifier, on_product.payment_date, calc_grad=True)
on_expected_pv = NOTIONAL * on_tau * (ON_LEVERAGE * on_expected_forward + ON_SPREAD) * on_df_pay

print('engine forward:', float(on_engine.forward_rate_.detach()), ' closed:', float(on_expected_forward.detach()))
print('engine PV     :', _to_float(on_engine.value), ' closed:', float(on_expected_pv.detach()))
assert abs(float(on_engine.forward_rate_.detach()) - float(on_expected_forward.detach())) < 1e-10
assert abs(_to_float(on_engine.value) - float(on_expected_pv.detach())) < 1e-6
print('PASSED')


engine forward: 0.042708817063390664  closed: 0.042708817063390664
engine PV     : 165098.3865335841  closed: 165098.3865335841
PASSED


### 2b. `pv01()` exact closed form -- `d(PV)/d(forward_rate)`, so it scales with `leverage`

In [11]:
on_pv01 = on_engine.pv01()
on_expected_pv01 = ON_LEVERAGE * NOTIONAL * on_tau * float(on_df_pay.detach()) * 1e-4
print('engine pv01:', on_pv01, ' expected:', on_expected_pv01)
assert abs(on_pv01 - on_expected_pv01) < 1e-6
print('PASSED')


engine pv01: 377.723300756785  expected: 377.7233007567851
PASSED


### 2c. Realized (on payment date) and matured branches -- needs daily fixings seeded across the whole period

In [12]:
def seed_daily_fixings(index, start, end, rate, biz_conv, hol_conv):
    name = index.index_name()
    if IndexFixingsManager().exists(name):
        qfRemoveIndexFixings(name)
    IndexFixingsManager()._map.setdefault(name, {})
    dates, d = [], start
    while d < end:
        if hol_conv.isBusinessDay(d):
            dates.append(d.ISO())
        d = Date(d + 1)
    qfInsertIndexFixing(name, dates, [rate] * len(dates))


seed_daily_fixings(sofr_index, on_product.effective_date, on_product.termination_date, 0.0430, on_biz_conv, on_hol_conv)

yc_on_pay = build_model(on_product.payment_date)
on_on_pay = ValuationEngineProductOvernightIndexCompositeCashflow(yc_on_pay, vpc, on_product, ValuationRequest.PV)
on_on_pay.calculate_value()
print('value_date == payment_date -> value:', on_on_pay.value, ' cash:', on_on_pay.cash, ' df:', on_on_pay.df_)
assert on_on_pay.df_ == 1.0 and on_on_pay.value == on_on_pay.cash and on_on_pay.cash != 0.0

matured_date = add_period(on_product.payment_date, Period('1D'), on_biz_conv, on_hol_conv)
yc_matured = build_model(matured_date)
on_matured = ValuationEngineProductOvernightIndexCompositeCashflow(yc_matured, vpc, on_product, ValuationRequest.PV)
on_matured.calculate_value()
print('value_date >  payment_date -> value:', on_matured.value, ' cash:', on_matured.cash)
assert on_matured.value == 0.0 and on_matured.cash == 0.0

# fully realized -- pv01/grad_at_par have no live graph left to differentiate
assert on_on_pay.pv01() == 0.0
print('PASSED')


63 fixing(s) for SOFR-1B is(are) inserted.
value_date == payment_date -> value: tensor(169555.7590, dtype=torch.float64)  cash: 169555.75901104786  df: 1.0
value_date >  payment_date -> value: 0.0  cash: 0.0
PASSED


### 2d. `get_risk()` vs. finite difference -- nonzero on `SOFR-1B` (projection) and `SOFR-1B-FLAT` (discounting), zero on `USD-LIBOR-BBA-3M`

In [13]:
grad = np.zeros(N_STATE)
on_engine.get_risk(gradient=grad)
print('SOFR-1B block      sum:', grad[SLICES['SOFR-1B']].sum())
print('USD-LIBOR-BBA-3M    sum:', grad[SLICES['USD-LIBOR-BBA-3M']].sum())
print('SOFR-1B-FLAT block  sum:', grad[SLICES['SOFR-1B-FLAT']].sum())
assert abs(grad[SLICES['USD-LIBOR-BBA-3M']].sum()) < 1e-8

base_pv = _to_float(on_engine.value)
fd_sofr = (bumped_pv(ValuationEngineProductOvernightIndexCompositeCashflow, on_product, sofr_bump=EPS) - base_pv) / EPS
fd_flat = (bumped_pv(ValuationEngineProductOvernightIndexCompositeCashflow, on_product, flat_bump=EPS) - base_pv) / EPS
print('finite-diff (SOFR-1B)     :', fd_sofr, ' vs analytic:', grad[SLICES['SOFR-1B']].sum())
print('finite-diff (SOFR-1B-FLAT):', fd_flat, ' vs analytic:', grad[SLICES['SOFR-1B-FLAT']].sum())
assert abs(fd_sofr - grad[SLICES['SOFR-1B']].sum()) / abs(fd_sofr) < REL_TOL
assert abs(fd_flat - grad[SLICES['SOFR-1B-FLAT']].sum()) / abs(fd_flat) < REL_TOL
print('PASSED')


SOFR-1B block      sum: 3709611.3165263617
USD-LIBOR-BBA-3M    sum: 0.0
SOFR-1B-FLAT block  sum: -56540.54333341922
finite-diff (SOFR-1B)     : 3709610.5128293857  vs analytic: 3709611.3165263617
finite-diff (SOFR-1B-FLAT): -56540.53366743028  vs analytic: -56540.54333341922
PASSED


## 3. Atomic index cashflow 2/3 -- `ValuationEngineProductIBORIndexCashflow`

`USD-LIBOR-BBA-3M`, a single native-tenor (3M) period, 10bp spread, `leverage=1.0`.

In [14]:
ib_biz_conv = libor_3m.payment_business_day_conv
ib_hol_conv = libor_3m.payment_holiday_conv
ib_termination = add_period(EFFECTIVE, libor_3m.term, ib_biz_conv, ib_hol_conv)
IB_SPREAD = 0.001

ib_product = ProductIBORIndexCashflow(
    EFFECTIVE, TermOrDate(ib_termination), PayOrReceive.RECEIVE, libor_3m,
    IB_SPREAD, Currency('USD'), NOTIONAL,
    payment_business_day_convention=ib_biz_conv, payment_holiday_convention=ib_hol_conv,
)
print('fixing_date:', ib_product.fixing_date, ' accrued:', ib_product.accrued)


fixing_date: August 17th, 2026  accrued: 0.25555555555555554


### 3a. PV vs. closed form (fully forward)

In [15]:
ib_engine = ValuationEngineProductIBORIndexCashflow(yc, vpc, ib_product, ValuationRequest.PV)
ib_engine.calculate_value()

ib_tau_native = accrued(EFFECTIVE, ib_termination, libor_3m.accrual_basis, ib_biz_conv, ib_hol_conv)
df_t0 = yc.discount_factor(libor_3m, EFFECTIVE, calc_grad=True)
df_te = yc.discount_factor(libor_3m, ib_termination, calc_grad=True)
ib_expected_forward = (df_t0 / df_te - 1.0) / ib_tau_native
ib_df_pay = yc.discount_factor(funding_identifier, ib_product.payment_date, calc_grad=True)
ib_tau = ib_product.accrued
ib_expected_pv = NOTIONAL * ib_tau * (ib_expected_forward + IB_SPREAD) * ib_df_pay

print('engine forward:', float(ib_engine.forward_rate_.detach()), ' closed:', float(ib_expected_forward.detach()))
print('engine PV     :', _to_float(ib_engine.value), ' closed:', float(ib_expected_pv.detach()))
assert abs(float(ib_engine.forward_rate_.detach()) - float(ib_expected_forward.detach())) < 1e-10
assert abs(_to_float(ib_engine.value) - float(ib_expected_pv.detach())) < 1e-6
print('PASSED')


engine forward: 0.08790018424108219  closed: 0.08790018424108219
engine PV     : 223864.47352951928  closed: 223864.47352951928
PASSED


### 3b. `pv01()` exact closed form

In [16]:
ib_pv01 = ib_engine.pv01()
ib_expected_pv01 = NOTIONAL * ib_tau * float(ib_df_pay.detach()) * 1e-4
print('engine pv01:', ib_pv01, ' expected:', ib_expected_pv01)
assert abs(ib_pv01 - ib_expected_pv01) < 1e-6
print('PASSED')


engine pv01: 251.8155338378567  expected: 251.8155338378567
PASSED


### 3c. Realized (on payment date) and matured branches -- one fixing at the period's own fixing date

In [17]:
def set_single_fixing(index, date, rate):
    name = index.index_name()
    if IndexFixingsManager().exists(name):
        qfRemoveIndexFixings(name)
    IndexFixingsManager()._map.setdefault(name, {})
    qfInsertIndexFixing(name, [date.ISO()], [rate])


set_single_fixing(libor_3m, ib_product.fixing_date, 0.0455)

yc_on_pay = build_model(ib_product.payment_date)
ib_on_pay = ValuationEngineProductIBORIndexCashflow(yc_on_pay, vpc, ib_product, ValuationRequest.PV)
ib_on_pay.calculate_value()
print('value_date == payment_date -> value:', ib_on_pay.value, ' cash:', ib_on_pay.cash, ' df:', ib_on_pay.df_)
assert ib_on_pay.df_ == 1.0 and ib_on_pay.value == ib_on_pay.cash and ib_on_pay.cash != 0.0

matured_date = add_period(ib_product.payment_date, Period('1D'), ib_biz_conv, ib_hol_conv)
yc_matured = build_model(matured_date)
ib_matured = ValuationEngineProductIBORIndexCashflow(yc_matured, vpc, ib_product, ValuationRequest.PV)
ib_matured.calculate_value()
print('value_date >  payment_date -> value:', ib_matured.value, ' cash:', ib_matured.cash)
assert ib_matured.value == 0.0 and ib_matured.cash == 0.0
assert ib_on_pay.pv01() == 0.0
print('PASSED')


1 fixing(s) for USD-LIBOR-BBA-3M is(are) inserted.
value_date == payment_date -> value: 118833.33333333333  cash: 118833.33333333333  df: 1.0
value_date >  payment_date -> value: 0.0  cash: 0.0
PASSED


### 3d. `get_risk()` vs. finite difference

All three blocks are nonzero here: `USD-LIBOR-BBA-3M` (direct projection), `SOFR-1B-FLAT`
(direct discounting), and `SOFR-1B` (indirectly, through *both* `USD-LIBOR-BBA-3M` and
`SOFR-1B-FLAT` being `REFERENCE`d off it -- see the note in 1d above). Each block is checked
against its own independent finite difference.

In [18]:
grad = np.zeros(N_STATE)
ib_engine.get_risk(gradient=grad)
print('SOFR-1B block      sum:', grad[SLICES['SOFR-1B']].sum())
print('USD-LIBOR-BBA-3M    sum:', grad[SLICES['USD-LIBOR-BBA-3M']].sum())
print('SOFR-1B-FLAT block  sum:', grad[SLICES['SOFR-1B-FLAT']].sum())

base_pv = _to_float(ib_engine.value)
fd_sofr = (bumped_pv(ValuationEngineProductIBORIndexCashflow, ib_product, sofr_bump=EPS) - base_pv) / EPS
fd_libor = (bumped_pv(ValuationEngineProductIBORIndexCashflow, ib_product, libor_bump=EPS) - base_pv) / EPS
fd_flat = (bumped_pv(ValuationEngineProductIBORIndexCashflow, ib_product, flat_bump=EPS) - base_pv) / EPS
print('finite-diff (SOFR-1B)         :', fd_sofr, ' vs analytic:', grad[SLICES['SOFR-1B']].sum())
print('finite-diff (USD-LIBOR-BBA-3M):', fd_libor, ' vs analytic:', grad[SLICES['USD-LIBOR-BBA-3M']].sum())
print('finite-diff (SOFR-1B-FLAT)    :', fd_flat, ' vs analytic:', grad[SLICES['SOFR-1B-FLAT']].sum())
assert abs(fd_sofr - grad[SLICES['SOFR-1B']].sum()) / abs(fd_sofr) < REL_TOL
assert abs(fd_libor - grad[SLICES['USD-LIBOR-BBA-3M']].sum()) / abs(fd_libor) < REL_TOL
assert abs(fd_flat - grad[SLICES['SOFR-1B-FLAT']].sum()) / abs(fd_flat) < REL_TOL
print('PASSED')


SOFR-1B block      sum: 2462785.544927889
USD-LIBOR-BBA-3M    sum: 2539451.46052019
SOFR-1B-FLAT block  sum: -76665.91559230113
finite-diff (SOFR-1B)         : 2462785.007606726  vs analytic: 2462785.544927889
finite-diff (USD-LIBOR-BBA-3M): 2539451.7819513567  vs analytic: 2539451.46052019
finite-diff (SOFR-1B-FLAT)    : -76665.90248118155  vs analytic: -76665.91559230113
PASSED


## 4. Atomic index cashflow 3/3 -- `ValuationEngineProductIBORCompoundingCashflow`

Three consecutive native 3M `USD-LIBOR-BBA-3M` periods (9M span) geometrically compounded,
5bp spread, `leverage=1.0`. Checked under both `SPREAD_EXCLUSIVE_COMPOUND` and `FLAT_COMPOUND`
-- with no product spread ever reaching the analytics layer (spread is applied once, at the
product level, for both methods), a fully forward-looking period has nothing to distinguish the
two, so they must agree exactly.

In [19]:
cp_termination = add_period(EFFECTIVE, Period('9M'), ib_biz_conv, ib_hol_conv)
CP_SPREAD = 0.0005

cp_products = {}
for method in [CompoundingMethod.SPREAD_EXCLUSIVE_COMPOUND, CompoundingMethod.FLAT_COMPOUND]:
    cp_products[method] = ProductIBORCompoundingCashflow(
        EFFECTIVE, TermOrDate(cp_termination), PayOrReceive.RECEIVE, libor_3m,
        CP_SPREAD, Currency('USD'), NOTIONAL, Period('3M'), 1.0,
        pay_date_or_payment_offset=TermOrDate('0D'),
        payment_business_day_convention=ib_biz_conv, payment_holiday_convention=ib_hol_conv,
        compounding_method=method,
    )


### 4a. PV vs. closed form, both compounding methods agree (fully forward)

In [20]:
cur, cp_bounds = EFFECTIVE, []
while cur < cp_termination:
    nxt = add_period(cur, libor_3m.term, ib_biz_conv, ib_hol_conv)
    cp_bounds.append((cur, nxt))
    cur = nxt

cp_df_first = yc.discount_factor(libor_3m, cp_bounds[0][0], calc_grad=True)
cp_df_last = yc.discount_factor(libor_3m, cp_bounds[-1][1], calc_grad=True)
cp_engines = {}
cp_values = {}
for method, product in cp_products.items():
    tau_total = product.accrued
    expected_forward = (cp_df_first / cp_df_last - 1.0) / tau_total
    df_pay = yc.discount_factor(funding_identifier, product.payment_date, calc_grad=True)
    expected_pv = NOTIONAL * tau_total * (expected_forward + CP_SPREAD) * df_pay

    engine = ValuationEngineProductIBORCompoundingCashflow(yc, vpc, product, ValuationRequest.PV)
    engine.calculate_value()
    cp_engines[method] = engine
    cp_values[method] = _to_float(engine.value)

    print(f'--- {method} ---')
    print('engine forward:', float(engine.forward_rate_.detach()), ' closed:', float(expected_forward.detach()))
    print('engine PV     :', _to_float(engine.value), ' closed:', float(expected_pv.detach()))
    assert abs(float(engine.forward_rate_.detach()) - float(expected_forward.detach())) < 1e-8
    assert abs(_to_float(engine.value) - float(expected_pv.detach())) < 1e-4

diff = abs(cp_values[CompoundingMethod.SPREAD_EXCLUSIVE_COMPOUND] - cp_values[CompoundingMethod.FLAT_COMPOUND])
print('PV difference between methods (should be ~0):', diff)
assert diff < 1e-4
print('PASSED')


--- CompoundingMethod.SPREAD_EXCLUSIVE_COMPOUND ---
engine forward: 0.09032092499762352  closed: 0.09032092499762352
engine PV     : 664196.7778209932  closed: 664196.7778209932
--- CompoundingMethod.FLAT_COMPOUND ---
engine forward: 0.09032092499762352  closed: 0.09032092499762352
engine PV     : 664196.7778209932  closed: 664196.7778209932
PV difference between methods (should be ~0): 0.0
PASSED


### 4b. `pv01()` exact closed form, both methods

In [21]:
cp_tau = cp_products[CompoundingMethod.SPREAD_EXCLUSIVE_COMPOUND].accrued
cp_df_pay = yc.discount_factor(funding_identifier, cp_products[CompoundingMethod.SPREAD_EXCLUSIVE_COMPOUND].payment_date, calc_grad=True)
cp_expected_pv01 = NOTIONAL * cp_tau * float(cp_df_pay.detach()) * 1e-4

for method, engine in cp_engines.items():
    pv01 = engine.pv01()
    print(f'{method} pv01:', pv01, ' expected:', cp_expected_pv01)
    assert abs(pv01 - cp_expected_pv01) < 1e-4
print('PASSED')


CompoundingMethod.SPREAD_EXCLUSIVE_COMPOUND pv01: 731.3257135823853  expected: 731.3257135823853
CompoundingMethod.FLAT_COMPOUND pv01: 731.3257135823853  expected: 731.3257135823853
PASSED


### 4c. Realized (on payment date) and matured branches -- fixings seeded at each sub-period start

In [22]:
cp_sub_starts = [b[0] for b in cp_bounds]
set_single_fixing(libor_3m, cp_sub_starts[0], 0.0450)  # overwritten below to hold all 3
name = libor_3m.index_name()
qfRemoveIndexFixings(name)
IndexFixingsManager()._map.setdefault(name, {})
qfInsertIndexFixing(name, [d.ISO() for d in cp_sub_starts], [0.0450, 0.0452, 0.0455])

cp_product = cp_products[CompoundingMethod.SPREAD_EXCLUSIVE_COMPOUND]
yc_on_pay = build_model(cp_product.payment_date)
cp_on_pay = ValuationEngineProductIBORCompoundingCashflow(yc_on_pay, vpc, cp_product, ValuationRequest.PV)
cp_on_pay.calculate_value()
print('value_date == payment_date -> value:', cp_on_pay.value, ' cash:', cp_on_pay.cash, ' df:', cp_on_pay.df_)
assert cp_on_pay.df_ == 1.0 and cp_on_pay.value == cp_on_pay.cash and cp_on_pay.cash != 0.0

matured_date = add_period(cp_product.payment_date, Period('1D'), ib_biz_conv, ib_hol_conv)
yc_matured = build_model(matured_date)
cp_matured = ValuationEngineProductIBORCompoundingCashflow(yc_matured, vpc, cp_product, ValuationRequest.PV)
cp_matured.calculate_value()
print('value_date >  payment_date -> value:', cp_matured.value, ' cash:', cp_matured.cash)
assert cp_matured.value == 0.0 and cp_matured.cash == 0.0
assert cp_on_pay.pv01() == 0.0
print('PASSED')


The fixings of USD-LIBOR-BBA-3M are all removed.
1 fixing(s) for USD-LIBOR-BBA-3M is(are) inserted.
The fixings of USD-LIBOR-BBA-3M are all removed.
3 fixing(s) for USD-LIBOR-BBA-3M is(are) inserted.
value_date == payment_date -> value: tensor(350725.1389, dtype=torch.float64)  cash: 350725.1389173789  df: 1.0
value_date >  payment_date -> value: 0.0  cash: 0.0
PASSED


### 4d. `get_risk()` vs. finite difference

Same three-nonzero-block shape as 3d.

In [23]:
cp_engine = cp_engines[CompoundingMethod.SPREAD_EXCLUSIVE_COMPOUND]
grad = np.zeros(N_STATE)
cp_engine.get_risk(gradient=grad)
print('SOFR-1B block      sum:', grad[SLICES['SOFR-1B']].sum())
print('USD-LIBOR-BBA-3M    sum:', grad[SLICES['USD-LIBOR-BBA-3M']].sum())
print('SOFR-1B-FLAT block  sum:', grad[SLICES['SOFR-1B-FLAT']].sum())

base_pv = _to_float(cp_engine.value)
fd_sofr = (bumped_pv(ValuationEngineProductIBORCompoundingCashflow, cp_product, sofr_bump=EPS) - base_pv) / EPS
fd_libor = (bumped_pv(ValuationEngineProductIBORCompoundingCashflow, cp_product, libor_bump=EPS) - base_pv) / EPS
fd_flat = (bumped_pv(ValuationEngineProductIBORCompoundingCashflow, cp_product, flat_bump=EPS) - base_pv) / EPS
print('finite-diff (SOFR-1B)         :', fd_sofr, ' vs analytic:', grad[SLICES['SOFR-1B']].sum())
print('finite-diff (USD-LIBOR-BBA-3M):', fd_libor, ' vs analytic:', grad[SLICES['USD-LIBOR-BBA-3M']].sum())
print('finite-diff (SOFR-1B-FLAT)    :', fd_flat, ' vs analytic:', grad[SLICES['SOFR-1B-FLAT']].sum())
assert abs(fd_sofr - grad[SLICES['SOFR-1B']].sum()) / abs(fd_sofr) < REL_TOL
assert abs(fd_libor - grad[SLICES['USD-LIBOR-BBA-3M']].sum()) / abs(fd_libor) < REL_TOL
assert abs(fd_flat - grad[SLICES['SOFR-1B-FLAT']].sum()) / abs(fd_flat) < REL_TOL
print('PASSED')


SOFR-1B block      sum: 7150289.905834121
USD-LIBOR-BBA-3M    sum: 7707123.368884049
SOFR-1B-FLAT block  sum: -556833.4630499285
finite-diff (SOFR-1B)         : 7150286.559481174  vs analytic: 7150289.905834121
finite-diff (USD-LIBOR-BBA-3M): 7707126.252586022  vs analytic: 7707123.368884049
finite-diff (SOFR-1B-FLAT)    : -556833.2297261804  vs analytic: -556833.4630499285
PASSED


## 5. `ValuationEngineProductInterestRateStream`

A full leg -- a `ValuationEngineProductPortfolio` of per-period cashflows sharing one
`fixed_rate_or_spread`. Two streams: a 1Y quarterly **fixed** leg (4.5% coupon) and a 1Y
quarterly **floating** leg (`USD-SOFR-COMPOUND`, 25bp spread) -- both built off the same
`EFFECTIVE` date used above.

In [24]:
fixed_leg = ProductInterestRateStream(
    EFFECTIVE, TermOrDate('1Y'), PayOrReceive.RECEIVE, 0.045,
    Currency('USD'), NOTIONAL, on_biz_conv, on_hol_conv,
    accrual_period=Period('3M'),
    accrual_basis=AccrualBasis.new('ACTUAL/360'),
)
float_leg = ProductInterestRateStream(
    EFFECTIVE, TermOrDate('1Y'), PayOrReceive.RECEIVE, 0.0025,
    Currency('USD'), NOTIONAL, on_biz_conv, on_hol_conv,
    index=sofr_composite,
    accrual_period=Period('3M'),
)
print('fixed leg cashflows  :', fixed_leg.num_cashflows())
print('float leg cashflows  :', float_leg.num_cashflows())


fixed leg cashflows  : 4
float leg cashflows  : 4


### 5a. PV vs. an independent per-cashflow sum, at three different value dates

`value_date` sweeps from before the leg starts (all 4 cashflows forward-looking), to partway
through its life (the first cashflow matured, the rest still forward), to fully after the
leg's last payment (fully matured, PV/cash both zero).

`ValuationEngineProductOvernightIndexCompositeCashflow.calculate_value()` always runs the full
daily-compounding analytics engine first (to get `forward_rate_`) and only *afterward* decides
whether the settlement is zero/cash/discounted based on `value_date_` vs. `payment_date_` --
so even a cashflow that's about to be discarded as "matured" still needs every daily fixing
across its own accrual window seeded, or the fixings lookup itself raises. Seed the whole
floating leg's span with a flat 4.30% daily fixing series upfront to cover every cashflow at
every value date used below.

In [25]:
last_termination = fixed_leg.cashflow(fixed_leg.num_cashflows() - 1).termination_date
seed_daily_fixings(sofr_index, EFFECTIVE, last_termination, 0.0430, on_biz_conv, on_hol_conv)


def manual_leg_pv(model, product, engine_cls):
    total = 0.0
    for prod, weight in product.elements_:
        e = engine_cls(model, vpc, prod, ValuationRequest.PV)
        e.calculate_value()
        total += weight * _to_float(e.value)
    return total


# (i) fully forward
for leg, engine_cls in [(fixed_leg, ValuationEngineProductFixedAccrued),
                         (float_leg, ValuationEngineProductOvernightIndexCompositeCashflow)]:
    eng = ValuationEngineProductInterestRateStream(yc, vpc, leg, ValuationRequest.PV)
    eng.calculate_value()
    manual = manual_leg_pv(yc, leg, engine_cls)
    print(f'{leg.index.index_name() if leg.index else "FIXED"} leg, fully forward -- engine PV: {_to_float(eng.value):.4f}  manual PV: {manual:.4f}')
    assert abs(_to_float(eng.value) - manual) < 1e-4

# (ii) partway through -- value_date after the first cashflow's payment date but before the
# second's; the first element is matured (contributes 0), the rest are still forward.
first_payment = fixed_leg.cashflow(0).payment_date
mid_date = add_period(first_payment, Period('1D'), on_biz_conv, on_hol_conv)
yc_mid = build_model(mid_date)
for leg, engine_cls in [(fixed_leg, ValuationEngineProductFixedAccrued),
                         (float_leg, ValuationEngineProductOvernightIndexCompositeCashflow)]:
    eng = ValuationEngineProductInterestRateStream(yc_mid, vpc, leg, ValuationRequest.PV)
    eng.calculate_value()
    manual = manual_leg_pv(yc_mid, leg, engine_cls)
    print(f'{leg.index.index_name() if leg.index else "FIXED"} leg, mid-life -- engine PV: {_to_float(eng.value):.4f}  manual PV: {manual:.4f}')
    assert abs(_to_float(eng.value) - manual) < 1e-4

# (iii) fully matured -- value_date after the last cashflow's payment date
last_payment = fixed_leg.cashflow(fixed_leg.num_cashflows() - 1).payment_date
matured_date = add_period(last_payment, Period('1D'), on_biz_conv, on_hol_conv)
yc_matured_leg = build_model(matured_date)
for leg in [fixed_leg, float_leg]:
    eng = ValuationEngineProductInterestRateStream(yc_matured_leg, vpc, leg, ValuationRequest.PV)
    eng.calculate_value()
    print(f'{leg.index.index_name() if leg.index else "FIXED"} leg, fully matured -- value: {eng.value}  cash: {eng.cash}')
    assert eng.value == 0.0 and eng.cash == 0.0
print('PASSED')


The fixings of SOFR-1B are all removed.
249 fixing(s) for SOFR-1B is(are) inserted.
FIXED leg, fully forward -- engine PV: 442382.1539  manual PV: 442382.1539
USD-SOFR-COMPOUND leg, fully forward -- engine PV: 447486.4047  manual PV: 447486.4047
FIXED leg, mid-life -- engine PV: 334038.5007  manual PV: 334038.5007
USD-SOFR-COMPOUND leg, mid-life -- engine PV: 336828.1090  manual PV: 336828.1090
FIXED leg, fully matured -- value: 0.0  cash: 0.0


USD-SOFR-COMPOUND leg, fully matured -- value: 0.0  cash: 0.0
PASSED


### 5b. `par_rate_or_spread()` -- solving for the rate/spread that zeros the leg's own PV

For the fixed leg this is trivially `0.0` in isolation (a fixed leg's PV is exactly
proportional to its own coupon, with no other term -- "par" only becomes a non-trivial
quantity relative to a second, offsetting leg, e.g. inside a swap). For the floating leg it's
the genuine par spread level; rebuilding the leg at that spread must zero its PV.

In [26]:
fixed_eng = ValuationEngineProductInterestRateStream(yc, vpc, fixed_leg, ValuationRequest.PV)
fixed_eng.calculate_value()
fixed_par = fixed_eng.par_rate_or_spread()
print('fixed leg par rate (expected exactly 0.0):', fixed_par)
assert fixed_par == 0.0

float_eng = ValuationEngineProductInterestRateStream(yc, vpc, float_leg, ValuationRequest.PV)
float_eng.calculate_value()
float_par = float_eng.par_rate_or_spread()
print('float leg par spread:', float_par)

float_leg_at_par = ProductInterestRateStream(
    EFFECTIVE, TermOrDate('1Y'), PayOrReceive.RECEIVE, float_par,
    Currency('USD'), NOTIONAL, on_biz_conv, on_hol_conv,
    index=sofr_composite,
    accrual_period=Period('3M'),
)
float_eng_at_par = ValuationEngineProductInterestRateStream(yc, vpc, float_leg_at_par, ValuationRequest.PV)
float_eng_at_par.calculate_value()
print('float leg PV at par spread (should be ~0):', _to_float(float_eng_at_par.value))
assert abs(_to_float(float_eng_at_par.value)) < 1e-3
print('PASSED')


fixed leg par rate (expected exactly 0.0): 0.0
float leg par spread: -0.043019214636875594
float leg PV at par spread (should be ~0): 9.15179043659009e-11
PASSED


### 5c. `pv01()` vs. finite difference (bump the shared fixed_rate/spread by 1bp and reprice the leg)

In [27]:
def leg_pv_at_rate(product_ctor, rate):
    leg = product_ctor(rate)
    eng = ValuationEngineProductInterestRateStream(yc, vpc, leg, ValuationRequest.PV)
    eng.calculate_value()
    return _to_float(eng.value)


make_fixed_leg = lambda r: ProductInterestRateStream(
    EFFECTIVE, TermOrDate('1Y'), PayOrReceive.RECEIVE, r,
    Currency('USD'), NOTIONAL, on_biz_conv, on_hol_conv,
    accrual_period=Period('3M'), accrual_basis=AccrualBasis.new('ACTUAL/360'),
)
make_float_leg = lambda r: ProductInterestRateStream(
    EFFECTIVE, TermOrDate('1Y'), PayOrReceive.RECEIVE, r,
    Currency('USD'), NOTIONAL, on_biz_conv, on_hol_conv,
    index=sofr_composite, accrual_period=Period('3M'),
)

for label, ctor, engine, base_rate in [
    ('fixed', make_fixed_leg, fixed_eng, 0.045),
    ('float', make_float_leg, float_eng, 0.0025),
]:
    pv01 = engine.pv01()
    fd_pv01 = leg_pv_at_rate(ctor, base_rate + 1e-4) - leg_pv_at_rate(ctor, base_rate)
    print(f'{label} leg pv01: {pv01:.4f}   finite-diff: {fd_pv01:.4f}')
    assert abs(pv01 - fd_pv01) < 1e-3
print('PASSED')


fixed leg pv01: 983.0715   finite-diff: 983.0715
float leg pv01: 983.0715   finite-diff: 983.0715
PASSED


### 5d. `get_risk()` (inherited from `ValuationEngineProductPortfolio`) vs. finite difference

`USD-LIBOR-BBA-3M` never enters either leg, so that block must be exactly zero for both. Both
legs discount through `SOFR-1B-FLAT`, which is `REFERENCE`d off `SOFR-1B`, so *both* legs --
including the fixed one -- carry a real nonzero `SOFR-1B` gradient block too (same point as
1d above); the floating leg's `SOFR-1B` block is additionally larger since it also projects
directly off `SOFR-1B`.

In [28]:
for label, leg, engine in [('fixed', fixed_leg, fixed_eng), ('float', float_leg, float_eng)]:
    grad = np.zeros(N_STATE)
    engine.get_risk(gradient=grad)
    print(f'--- {label} leg ---')
    print('SOFR-1B block      sum:', grad[SLICES['SOFR-1B']].sum())
    print('USD-LIBOR-BBA-3M    sum:', grad[SLICES['USD-LIBOR-BBA-3M']].sum())
    print('SOFR-1B-FLAT block  sum:', grad[SLICES['SOFR-1B-FLAT']].sum())
    assert abs(grad[SLICES['USD-LIBOR-BBA-3M']].sum()) < 1e-8

    base_pv = _to_float(engine.value)

    def bumped_leg_pv(**bumps):
        yc_bumped = build_model(VALUE_DATE, **bumps)
        eng = ValuationEngineProductInterestRateStream(yc_bumped, vpc, leg, ValuationRequest.PV)
        eng.calculate_value()
        return _to_float(eng.value)

    fd_sofr = (bumped_leg_pv(sofr_bump=EPS) - base_pv) / EPS
    fd_flat = (bumped_leg_pv(flat_bump=EPS) - base_pv) / EPS
    print('finite-diff (SOFR-1B)     :', fd_sofr, ' vs analytic:', grad[SLICES['SOFR-1B']].sum())
    print('finite-diff (SOFR-1B-FLAT):', fd_flat, ' vs analytic:', grad[SLICES['SOFR-1B-FLAT']].sum())
    assert abs(fd_sofr - grad[SLICES['SOFR-1B']].sum()) / abs(fd_sofr) < REL_TOL
    assert abs(fd_flat - grad[SLICES['SOFR-1B-FLAT']].sum()) / abs(fd_flat) < REL_TOL
print('PASSED')


--- fixed leg ---
SOFR-1B block      sum: -314998.17979925475
USD-LIBOR-BBA-3M    sum: 0.0
SOFR-1B-FLAT block  sum: -314998.17979925475
finite-diff (SOFR-1B)     : -314998.05045314133  vs analytic: -314998.17979925475
finite-diff (SOFR-1B-FLAT): -314998.05045314133  vs analytic: -314998.17979925475
--- float leg ---
SOFR-1B block      sum: 9482553.578345941
USD-LIBOR-BBA-3M    sum: 0.0
SOFR-1B-FLAT block  sum: -319244.80478431337
finite-diff (SOFR-1B)     : 9482547.957333736  vs analytic: 9482553.578345941
finite-diff (SOFR-1B-FLAT): -319244.67347562313  vs analytic: -319244.80478431337
PASSED


### 5e. `RiskValParam` / `product_risk` / `get_risk_report()`

`ValuationEngineProductInterestRateStream` *is* a `ValuationEngineProductPortfolio` (see the
class hierarchy note in section 5's own header) -- `fixed_leg` above is a real
`ProductPortfolio` with 4 quarterly `ProductFixedAccrued` elements (`fixed_leg.elements_`),
each carrying weight `1.0` (every element of a `ProductInterestRateStream` is built with
weight `1.0` -- see `ProductInterestRateStream.__init__` in `product/linear_products.py`).
This section exercises the new `RiskValParam` vpc entry and the `product_risk`/
`get_risk_report()` surface it drives on `ValuationEngineProductPortfolio`, reusing this
already-built leg rather than constructing a fresh portfolio fixture.

In [29]:
from fixedincomelib.valuation.valuation_parameters import RiskValParam

def make_vpc(risk_level=None):
    vps = [
        FundingIndexParameter({
            'FUNDING INDEX': 'SOFR-1B-FLAT',
            'CURRENCIES': '',
            'FUNDING INDICES': '',
            'UNDERLYING FUNDING INDEX': '',
        }),
    ]
    if risk_level is not None:
        vps.append(RiskValParam({'LEVEL': risk_level}))
    return ValuationParametersCollection(vps)

vpc_no_risk_param = make_vpc(risk_level=None)
vpc_portfolio = make_vpc(risk_level='PORTFOLIO')
vpc_product = make_vpc(risk_level='PRODUCT')

print('fixed leg element count:', len(fixed_leg.elements_))
assert len(fixed_leg.elements_) >= 2
assert not vpc_no_risk_param.has_vp_type(RiskValParam._vp_type)
assert vpc_portfolio.get_vp_from_build_method_collection(RiskValParam._vp_type).level == 'PORTFOLIO'
assert vpc_product.get_vp_from_build_method_collection(RiskValParam._vp_type).level == 'PRODUCT'

fixed leg element count: 4


#### 5e-i. `get_risk()`'s aggregated output equals the sum of `product_risk` entries

In [30]:
eng_no_rp = ValuationEngineProductInterestRateStream(yc, vpc_no_risk_param, fixed_leg, ValuationRequest.PV)
eng_no_rp.calculate_value()
agg_grad = np.zeros(N_STATE)
eng_no_rp.get_risk(gradient=agg_grad)

assert len(eng_no_rp.product_risk) == len(fixed_leg.elements_)
summed_product_risk = np.sum(np.stack(eng_no_rp.product_risk), axis=0)
print('max abs diff (sum(product_risk) vs aggregated get_risk):', np.max(np.abs(summed_product_risk - agg_grad)))
assert np.allclose(summed_product_risk, agg_grad, atol=1e-8)
print('PASSED')

max abs diff (sum(product_risk) vs aggregated get_risk): 0.0
PASSED


#### 5e-ii. `get_risk_report()` with no `RiskValParam` in the vpc defaults to `PORTFOLIO` --
same aggregated array `get_risk()` produces

In [31]:
report_default = eng_no_rp.get_risk_report()
print('report type (should be the aggregated ndarray, not a list):', type(report_default))
assert isinstance(report_default, np.ndarray)
assert np.allclose(report_default, agg_grad, atol=1e-8)
print('PASSED')

report type (should be the aggregated ndarray, not a list): <class 'numpy.ndarray'>
PASSED


#### 5e-iii. Explicit `RiskValParam({'LEVEL': 'PORTFOLIO'})` matches the no-`RiskValParam` default

In [32]:
eng_portfolio = ValuationEngineProductInterestRateStream(yc, vpc_portfolio, fixed_leg, ValuationRequest.PV)
eng_portfolio.calculate_value()
report_portfolio = eng_portfolio.get_risk_report()
assert isinstance(report_portfolio, np.ndarray)
assert np.allclose(report_portfolio, agg_grad, atol=1e-8)
print('PASSED')

PASSED


#### 5e-iv. `RiskValParam({'LEVEL': 'PRODUCT'})` -- `get_risk_report()` returns the
per-element breakdown, matching `product_risk`; summing it reproduces the aggregated array;
and element order matches `product.elements_` order (checked against independently-built
per-cashflow engines, not just internal self-consistency)

In [33]:
eng_product = ValuationEngineProductInterestRateStream(yc, vpc_product, fixed_leg, ValuationRequest.PV)
eng_product.calculate_value()
report_product = eng_product.get_risk_report()

assert isinstance(report_product, list)
assert len(report_product) == len(fixed_leg.elements_)
for a, b in zip(report_product, eng_product.product_risk):
    assert np.array_equal(a, b)

summed_report = np.sum(np.stack(report_product), axis=0)
agg_grad_product = np.zeros(N_STATE)
eng_product.get_risk(gradient=agg_grad_product)
print('max abs diff (sum(get_risk_report PRODUCT) vs aggregated get_risk):', np.max(np.abs(summed_report - agg_grad_product)))
assert np.allclose(summed_report, agg_grad_product, atol=1e-8)

# order check: every ProductInterestRateStream element carries weight 1.0 (see
# product/linear_products.py), so product_risk[i] should equal that element's own
# independently-built engine risk exactly, in fixed_leg.elements_ order.
for i, (prod_i, weight_i) in enumerate(fixed_leg.elements_):
    assert weight_i == 1.0
    manual_engine = ValuationEngineProductFixedAccrued(yc, vpc_product, prod_i, ValuationRequest.PV)
    manual_engine.calculate_value()
    manual_grad = np.zeros(N_STATE)
    manual_engine.get_risk(gradient=manual_grad)
    diff = np.max(np.abs(manual_grad - eng_product.product_risk[i]))
    print(f'element {i}: max abs diff vs independently-built engine:', diff)
    assert np.allclose(manual_grad, eng_product.product_risk[i], atol=1e-8)
print('PASSED')

max abs diff (sum(get_risk_report PRODUCT) vs aggregated get_risk): 0.0
element 0: max abs diff vs independently-built engine: 0.0
element 1: max abs diff vs independently-built engine: 0.0
element 2: max abs diff vs independently-built engine: 0.0
element 3: max abs diff vs independently-built engine: 0.0
PASSED


#### 5e-v. `get_risk()` does not accumulate -- calling it twice into fresh arrays gives
identical results each time (no hidden state carried across calls)

In [34]:
grad_first = np.zeros(N_STATE)
eng_product.get_risk(gradient=grad_first)
grad_second = np.zeros(N_STATE)
eng_product.get_risk(gradient=grad_second)
assert np.array_equal(grad_first, grad_second)
print('PASSED (get_risk does not accumulate across repeated calls)')

PASSED (get_risk does not accumulate across repeated calls)


## Additional setup for sections 6-11

Imports for the six remaining `ValuationEngineProduct*` classes in `valuation_engine.py` (two-leg swaps, the future, the FRA/fixing, and the cash deposit) and one extra fixing-seeding helper (`seed_period_fixings`) for indices that fix once per accrual period (LIBOR-3M legs), complementing `seed_daily_fixings` (overnight, section 2) and `set_single_fixing` (single period, section 3) already defined above.

In [35]:
from fixedincomelib.yield_curve.valuation_engine import (
    ValuationEngineProductOvernightIndexSwap,
    ValuationEngineProductOvernightIndexBasisSwap,
    ValuationEngineProductOISBasisSwap,
    ValuationEngineProductOvernightIndexFuture,
    ValuationEngineProductFRAOrFixing,
    ValuationEngineProductCashDeposit,
)
from fixedincomelib.product.linear_products import (
    ProductOvernightIndexSwap,
    ProductOvernightIndexBasisSwap,
    ProductOISBasisSwap,
    ProductOvernightIndexFuture,
    ProductFRAOrFixing,
    ProductCashDeposit,
)


def seed_period_fixings(index, dates, rate):
    name = index.index_name()
    if IndexFixingsManager().exists(name):
        qfRemoveIndexFixings(name)
    IndexFixingsManager()._map.setdefault(name, {})
    qfInsertIndexFixing(name, [d.ISO() for d in dates], [rate] * len(dates))


print('Additional imports for sections 6-11 loaded.')

Additional imports for sections 6-11 loaded.


## 6. `ValuationEngineProductOvernightIndexSwap`

A fixed-vs-floating OIS on `USD-SOFR-COMPOUND`: 1Y, quarterly, RECEIVE fixed at 4.5% vs PAY SOFR-compound flat (no spread). Built directly as `ProductOvernightIndexSwap` (the product wires up its own `fixed_leg`/`floating_leg` `ProductInterestRateStream`s with opposite `pay_or_rec`, unlike section 5's ad hoc pairing). The engine wraps two `ValuationEngineProductInterestRateStream` legs whose PVs are therefore already sign-consistent -- `value_ = fixed_leg_engine_.value_ + floating_leg_engine_.value_`, no extra sign layer on top.

In [36]:
OIS_FIXED_RATE = 0.045
ois_product = ProductOvernightIndexSwap(
    EFFECTIVE, TermOrDate('1Y'), sofr_composite, OIS_FIXED_RATE, PayOrReceive.RECEIVE,
    NOTIONAL, Period('3M'), AccrualBasis.new('ACTUAL/360'), on_biz_conv, on_hol_conv,
)
print('fixed leg pay_or_rec :', ois_product.fixed_leg.pay_or_rec.to_string())
print('float leg pay_or_rec :', ois_product.floating_leg.pay_or_rec.to_string())
print('fixed leg cashflows  :', ois_product.fixed_leg.num_cashflows())
print('float leg cashflows  :', ois_product.floating_leg.num_cashflows())

fixed leg pay_or_rec : receive
float leg pay_or_rec : pay
fixed leg cashflows  : 4
float leg cashflows  : 4


### 6a. PV vs. independent closed form (sum of each leg's own per-cashflow closed forms, the same `manual_leg_pv` oracle used in section 5a)

In [37]:
ois_engine = ValuationEngineProductOvernightIndexSwap(yc, vpc, ois_product, ValuationRequest.PV)
ois_engine.calculate_value()

manual_ois_pv = (
    manual_leg_pv(yc, ois_product.fixed_leg, ValuationEngineProductFixedAccrued)
    + manual_leg_pv(yc, ois_product.floating_leg, ValuationEngineProductOvernightIndexCompositeCashflow)
)
print('engine PV:', _to_float(ois_engine.value), ' manual PV:', manual_ois_pv)
assert abs(_to_float(ois_engine.value) - manual_ois_pv) < 1e-4
print('PASSED')

engine PV: 19472.53545097151  manual PV: 19472.53545097151
PASSED


### 6b. `par_rate_or_spread()` (PV at the solved par rate is ~0) and `pv01()` vs. finite difference (bump the swap's fixed rate by 1bp and reprice the whole swap)

In [38]:
def ois_pv_at_rate(rate):
    p = ProductOvernightIndexSwap(
        EFFECTIVE, TermOrDate('1Y'), sofr_composite, rate, PayOrReceive.RECEIVE,
        NOTIONAL, Period('3M'), AccrualBasis.new('ACTUAL/360'), on_biz_conv, on_hol_conv,
    )
    e = ValuationEngineProductOvernightIndexSwap(yc, vpc, p, ValuationRequest.PV)
    e.calculate_value()
    return _to_float(e.value)

par = ois_engine.par_rate_or_spread()
pv_at_par = ois_pv_at_rate(par)
print('par fixed rate:', par, ' PV at par (should be ~0):', pv_at_par)
assert abs(pv_at_par) < 1e-2

pv01 = ois_engine.pv01()
fd_pv01 = ois_pv_at_rate(OIS_FIXED_RATE + 1e-4) - ois_pv_at_rate(OIS_FIXED_RATE)
print('engine pv01:', pv01, ' finite-diff:', fd_pv01)
assert abs(pv01 - fd_pv01) < 1e-3
print('PASSED')

par fixed rate: 0.04301921463687561  PV at par (should be ~0): 0.0


engine pv01: 983.0714530450934  finite-diff: 983.0714530451223
PASSED


### 6c. Settlement-date branch: fully matured (value_date after both legs' last payment)

In [39]:
ois_last_fixed_payment = ois_product.fixed_leg.cashflow(ois_product.fixed_leg.num_cashflows() - 1).payment_date
ois_last_float_payment = ois_product.floating_leg.cashflow(ois_product.floating_leg.num_cashflows() - 1).payment_date
ois_last_payment = max(ois_last_fixed_payment, ois_last_float_payment)
ois_matured_date = add_period(ois_last_payment, Period('1D'), on_biz_conv, on_hol_conv)

seed_daily_fixings(sofr_index, ois_product.floating_leg.effective_date, ois_product.floating_leg.termination_date, 0.0430, on_biz_conv, on_hol_conv)

yc_ois_matured = build_model(ois_matured_date)
ois_matured_engine = ValuationEngineProductOvernightIndexSwap(yc_ois_matured, vpc, ois_product, ValuationRequest.PV)
ois_matured_engine.calculate_value()
print('fully matured -- value:', ois_matured_engine.value, ' cash:', ois_matured_engine.cash)
assert ois_matured_engine.value == 0.0 and ois_matured_engine.cash == 0.0
print('PASSED')

The fixings of SOFR-1B are all removed.
249 fixing(s) for SOFR-1B is(are) inserted.


fully matured -- value: 0.0  cash: 0.0
PASSED


### 6d. `get_risk()` (overridden two-leg accumulation, not the base single-tensor default) vs. finite difference

In [40]:
grad = np.zeros(N_STATE)
ois_engine.get_risk(gradient=grad)
print('SOFR-1B block      sum:', grad[SLICES['SOFR-1B']].sum())
print('USD-LIBOR-BBA-3M    sum:', grad[SLICES['USD-LIBOR-BBA-3M']].sum())
print('SOFR-1B-FLAT block  sum:', grad[SLICES['SOFR-1B-FLAT']].sum())
assert abs(grad[SLICES['USD-LIBOR-BBA-3M']].sum()) < 1e-8

base_pv = _to_float(ois_engine.value)
fd_sofr = (bumped_pv(ValuationEngineProductOvernightIndexSwap, ois_product, sofr_bump=EPS) - base_pv) / EPS
fd_flat = (bumped_pv(ValuationEngineProductOvernightIndexSwap, ois_product, flat_bump=EPS) - base_pv) / EPS
print('finite-diff (SOFR-1B)     :', fd_sofr, ' vs analytic:', grad[SLICES['SOFR-1B']].sum())
print('finite-diff (SOFR-1B-FLAT):', fd_flat, ' vs analytic:', grad[SLICES['SOFR-1B-FLAT']].sum())
assert abs(fd_sofr - grad[SLICES['SOFR-1B']].sum()) / abs(fd_sofr) < REL_TOL
assert abs(fd_flat - grad[SLICES['SOFR-1B-FLAT']].sum()) / abs(fd_flat) < REL_TOL
print('PASSED')

SOFR-1B block      sum: -9815051.65702293
USD-LIBOR-BBA-3M    sum: 0.0
SOFR-1B-FLAT block  sum: -13253.273892677738


finite-diff (SOFR-1B)     : -9815045.899420511  vs analytic: -9815051.65702293
finite-diff (SOFR-1B-FLAT): -13253.268611151725  vs analytic: -13253.273892677738
PASSED


## 7. `ValuationEngineProductOvernightIndexBasisSwap`

Float-vs-float: `on_composite_index_leg` (`USD-SOFR-COMPOUND`, carries the quoted spread, RECEIVE) vs `ibor_index_leg` (`USD-LIBOR-BBA-3M`, `fixed_rate_or_spread=0`, PAY). Same two-leg-engine shape as section 6, but with no "fixed leg" -- the on-composite leg stands in for it in the `par_rate_or_spread()`/`pv01()` formulas since it's the only leg whose payoff depends on the spread being solved for.

In [41]:
ON_BASIS_SPREAD = 0.0012
onbs_product = ProductOvernightIndexBasisSwap(
    EFFECTIVE, TermOrDate('1Y'), sofr_composite, libor_3m, ON_BASIS_SPREAD,
    PayOrReceive.RECEIVE, NOTIONAL, Period('3M'), on_biz_conv, on_hol_conv,
)
print('on leg pay_or_rec   :', onbs_product.on_composite_index_leg.pay_or_rec.to_string())
print('ibor leg pay_or_rec :', onbs_product.ibor_index_leg.pay_or_rec.to_string())
print('on leg cashflows    :', onbs_product.on_composite_index_leg.num_cashflows())
print('ibor leg cashflows  :', onbs_product.ibor_index_leg.num_cashflows())

on leg pay_or_rec   : receive
ibor leg pay_or_rec : pay
on leg cashflows    : 4
ibor leg cashflows  : 4


### 7a. PV vs. independent closed form (sum of each leg's own per-cashflow closed forms)

In [42]:
onbs_engine = ValuationEngineProductOvernightIndexBasisSwap(yc, vpc, onbs_product, ValuationRequest.PV)
onbs_engine.calculate_value()

manual_onbs_pv = (
    manual_leg_pv(yc, onbs_product.on_composite_index_leg, ValuationEngineProductOvernightIndexCompositeCashflow)
    + manual_leg_pv(yc, onbs_product.ibor_index_leg, ValuationEngineProductIBORIndexCashflow)
)
print('engine PV:', _to_float(onbs_engine.value), ' manual PV:', manual_onbs_pv)
assert abs(_to_float(onbs_engine.value) - manual_onbs_pv) < 1e-4
print('PASSED')

engine PV:

 -435390.15331514925  manual PV: -435390.15331514925
PASSED


### 7b. `par_rate_or_spread()` and `pv01()` vs. finite difference (bump the spread by 1bp)

In [43]:
def onbs_pv_at_spread(spread):
    p = ProductOvernightIndexBasisSwap(
        EFFECTIVE, TermOrDate('1Y'), sofr_composite, libor_3m, spread,
        PayOrReceive.RECEIVE, NOTIONAL, Period('3M'), on_biz_conv, on_hol_conv,
    )
    e = ValuationEngineProductOvernightIndexBasisSwap(yc, vpc, p, ValuationRequest.PV)
    e.calculate_value()
    return _to_float(e.value)

par = onbs_engine.par_rate_or_spread()
pv_at_par = onbs_pv_at_spread(par)
print('par spread:', par, ' PV at par (should be ~0):', pv_at_par)
assert abs(pv_at_par) < 1e-2

pv01 = onbs_engine.pv01()
fd_pv01 = onbs_pv_at_spread(ON_BASIS_SPREAD + 1e-4) - onbs_pv_at_spread(ON_BASIS_SPREAD)
print('engine pv01:', pv01, ' finite-diff:', fd_pv01)
assert abs(pv01 - fd_pv01) < 1e-3
print('PASSED')

par spread: 0.04548875967932088  PV at par (should be ~0): -1.1641532182693481e-10


engine pv01: 983.0714530450934  finite-diff: 983.0714530450641
PASSED


### 7c. Settlement-date branch: fully matured (value_date after both legs' last payment) -- needs both an overnight fixing series (on leg) and per-period LIBOR fixings (ibor leg)

In [44]:
onbs_last_on_payment = onbs_product.on_composite_index_leg.cashflow(onbs_product.on_composite_index_leg.num_cashflows() - 1).payment_date
onbs_last_ibor_payment = onbs_product.ibor_index_leg.cashflow(onbs_product.ibor_index_leg.num_cashflows() - 1).payment_date
onbs_last_payment = max(onbs_last_on_payment, onbs_last_ibor_payment)
onbs_matured_date = add_period(onbs_last_payment, Period('1D'), on_biz_conv, on_hol_conv)

seed_daily_fixings(sofr_index, onbs_product.on_composite_index_leg.effective_date, onbs_product.on_composite_index_leg.termination_date, 0.0430, on_biz_conv, on_hol_conv)
onbs_ibor_fixing_dates = [onbs_product.ibor_index_leg.cashflow(i).fixing_date for i in range(onbs_product.ibor_index_leg.num_cashflows())]
seed_period_fixings(libor_3m, onbs_ibor_fixing_dates, 0.0455)

yc_onbs_matured = build_model(onbs_matured_date)
onbs_matured_engine = ValuationEngineProductOvernightIndexBasisSwap(yc_onbs_matured, vpc, onbs_product, ValuationRequest.PV)
onbs_matured_engine.calculate_value()
print('fully matured -- value:', onbs_matured_engine.value, ' cash:', onbs_matured_engine.cash)
assert onbs_matured_engine.value == 0.0 and onbs_matured_engine.cash == 0.0
print('PASSED')

The fixings of SOFR-1B are all removed.
249 fixing(s) for SOFR-1B is(are) inserted.
The fixings of USD-LIBOR-BBA-3M are all removed.
4 fixing(s) for USD-LIBOR-BBA-3M is(are) inserted.


fully matured -- value: 0.0  cash: 0.0
PASSED


### 7d. `get_risk()` vs. finite difference -- nonzero on all three components this time (`USD-LIBOR-BBA-3M` directly, via the ibor leg)

In [45]:
grad = np.zeros(N_STATE)
onbs_engine.get_risk(gradient=grad)
print('SOFR-1B block      sum:', grad[SLICES['SOFR-1B']].sum())
print('USD-LIBOR-BBA-3M    sum:', grad[SLICES['USD-LIBOR-BBA-3M']].sum())
print('SOFR-1B-FLAT block  sum:', grad[SLICES['SOFR-1B-FLAT']].sum())

base_pv = _to_float(onbs_engine.value)
fd_sofr = (bumped_pv(ValuationEngineProductOvernightIndexBasisSwap, onbs_product, sofr_bump=EPS) - base_pv) / EPS
fd_libor = (bumped_pv(ValuationEngineProductOvernightIndexBasisSwap, onbs_product, libor_bump=EPS) - base_pv) / EPS
fd_flat = (bumped_pv(ValuationEngineProductOvernightIndexBasisSwap, onbs_product, flat_bump=EPS) - base_pv) / EPS
print('finite-diff (SOFR-1B)         :', fd_sofr, ' vs analytic:', grad[SLICES['SOFR-1B']].sum())
print('finite-diff (USD-LIBOR-BBA-3M):', fd_libor, ' vs analytic:', grad[SLICES['USD-LIBOR-BBA-3M']].sum())
print('finite-diff (SOFR-1B-FLAT)    :', fd_flat, ' vs analytic:', grad[SLICES['SOFR-1B-FLAT']].sum())
assert abs(fd_sofr - grad[SLICES['SOFR-1B']].sum()) / abs(fd_sofr) < REL_TOL
assert abs(fd_libor - grad[SLICES['USD-LIBOR-BBA-3M']].sum()) / abs(fd_libor) < REL_TOL
assert abs(fd_flat - grad[SLICES['SOFR-1B-FLAT']].sum()) / abs(fd_flat) < REL_TOL
print('PASSED')

SOFR-1B block      sum: 198755.0831516058
USD-LIBOR-BBA-3M    sum: -9913620.28496983
SOFR-1B-FLAT block  sum: 310576.98499118607


finite-diff (SOFR-1B)         : 198755.02306967974  vs analytic: 198755.0831516058
finite-diff (USD-LIBOR-BBA-3M): -9913621.526095085  vs analytic: -9913620.28496983
finite-diff (SOFR-1B-FLAT)    : 310576.8571840599  vs analytic: 310576.98499118607
PASSED


## 8. `ValuationEngineProductOISBasisSwap`

Float-vs-float again, but both legs are overnight composite this time -- `basis_leg` (carries the quoted spread) vs. `reference_leg` (`fixed_rate_or_spread=0`). This curve only has one overnight projection component (`SOFR-1B`), so both legs are built off the *same* `USD-SOFR-COMPOUND` index here (a degenerate but perfectly valid basis swap for exercising the engine's mechanics) -- with identical schedules and an identical underlying compounded rate on both legs, the forward rate cancels between legs entirely and `PV(spread) = spread * basis_leg_engine_.annuity_` exactly, so `par_rate_or_spread()` should come out to (numerically) zero, a strong independent check on the formula.

In [46]:
OIS_BASIS_SPREAD = 0.0008
oisbs_product = ProductOISBasisSwap(
    EFFECTIVE, TermOrDate('1Y'), sofr_composite, sofr_composite, OIS_BASIS_SPREAD,
    PayOrReceive.RECEIVE, NOTIONAL, Period('3M'), on_biz_conv, on_hol_conv,
)
print('basis leg pay_or_rec     :', oisbs_product.basis_leg.pay_or_rec.to_string())
print('reference leg pay_or_rec :', oisbs_product.reference_leg.pay_or_rec.to_string())
print('basis leg cashflows      :', oisbs_product.basis_leg.num_cashflows())
print('reference leg cashflows  :', oisbs_product.reference_leg.num_cashflows())

basis leg pay_or_rec     : receive
reference leg pay_or_rec : pay
basis leg cashflows      : 4
reference leg cashflows  : 4


### 8a. PV vs. independent closed forms: the general (per-cashflow-sum) oracle, and, since both legs share the same index/schedule here, the degenerate `spread * annuity` oracle too

In [47]:
oisbs_engine = ValuationEngineProductOISBasisSwap(yc, vpc, oisbs_product, ValuationRequest.PV)
oisbs_engine.calculate_value()

manual_oisbs_pv = (
    manual_leg_pv(yc, oisbs_product.basis_leg, ValuationEngineProductOvernightIndexCompositeCashflow)
    + manual_leg_pv(yc, oisbs_product.reference_leg, ValuationEngineProductOvernightIndexCompositeCashflow)
)
print('engine PV:', _to_float(oisbs_engine.value), ' manual PV:', manual_oisbs_pv)
assert abs(_to_float(oisbs_engine.value) - manual_oisbs_pv) < 1e-4

basis_leg_engine = ValuationEngineProductInterestRateStream(yc, vpc, oisbs_product.basis_leg, ValuationRequest.PV)
basis_leg_engine.calculate_value()
degenerate_expected_pv = OIS_BASIS_SPREAD * _to_float(basis_leg_engine.annuity_)
print('degenerate closed form (spread * annuity):', degenerate_expected_pv)
assert abs(_to_float(oisbs_engine.value) - degenerate_expected_pv) < 1e-4
print('PASSED')

engine PV: 7864.571624360746  manual PV: 7864.571624360746


degenerate closed form (spread * annuity): 7864.571624360748
PASSED


### 8b. `par_rate_or_spread()` (should be ~0, since both legs share the same underlying rate) and `pv01()` vs. finite difference

In [48]:
def oisbs_pv_at_spread(spread):
    p = ProductOISBasisSwap(
        EFFECTIVE, TermOrDate('1Y'), sofr_composite, sofr_composite, spread,
        PayOrReceive.RECEIVE, NOTIONAL, Period('3M'), on_biz_conv, on_hol_conv,
    )
    e = ValuationEngineProductOISBasisSwap(yc, vpc, p, ValuationRequest.PV)
    e.calculate_value()
    return _to_float(e.value)

par = oisbs_engine.par_rate_or_spread()
print('par spread (expected ~0.0):', par)
assert abs(par) < 1e-8

pv01 = oisbs_engine.pv01()
fd_pv01 = oisbs_pv_at_spread(OIS_BASIS_SPREAD + 1e-4) - oisbs_pv_at_spread(OIS_BASIS_SPREAD)
print('engine pv01:', pv01, ' finite-diff:', fd_pv01)
assert abs(pv01 - fd_pv01) < 1e-3
print('PASSED')

par spread (expected ~0.0): 2.168404344971009e-19


engine pv01: 983.0714530450934  finite-diff: 983.0714530450641
PASSED


### 8c. Settlement-date branch: fully matured (value_date after both legs' last payment)

In [49]:
oisbs_last_basis_payment = oisbs_product.basis_leg.cashflow(oisbs_product.basis_leg.num_cashflows() - 1).payment_date
oisbs_last_reference_payment = oisbs_product.reference_leg.cashflow(oisbs_product.reference_leg.num_cashflows() - 1).payment_date
oisbs_last_payment = max(oisbs_last_basis_payment, oisbs_last_reference_payment)
oisbs_matured_date = add_period(oisbs_last_payment, Period('1D'), on_biz_conv, on_hol_conv)

seed_daily_fixings(sofr_index, oisbs_product.basis_leg.effective_date, oisbs_product.basis_leg.termination_date, 0.0430, on_biz_conv, on_hol_conv)

yc_oisbs_matured = build_model(oisbs_matured_date)
oisbs_matured_engine = ValuationEngineProductOISBasisSwap(yc_oisbs_matured, vpc, oisbs_product, ValuationRequest.PV)
oisbs_matured_engine.calculate_value()
print('fully matured -- value:', oisbs_matured_engine.value, ' cash:', oisbs_matured_engine.cash)
assert oisbs_matured_engine.value == 0.0 and oisbs_matured_engine.cash == 0.0
print('PASSED')

The fixings of SOFR-1B are all removed.
249 fixing(s) for SOFR-1B is(are) inserted.


fully matured -- value: 0.0  cash: 0.0
PASSED


### 8d. `get_risk()` vs. finite difference -- `USD-LIBOR-BBA-3M` never enters either leg

In [50]:
grad = np.zeros(N_STATE)
oisbs_engine.get_risk(gradient=grad)
print('SOFR-1B block      sum:', grad[SLICES['SOFR-1B']].sum())
print('USD-LIBOR-BBA-3M    sum:', grad[SLICES['USD-LIBOR-BBA-3M']].sum())
print('SOFR-1B-FLAT block  sum:', grad[SLICES['SOFR-1B-FLAT']].sum())
assert abs(grad[SLICES['USD-LIBOR-BBA-3M']].sum()) < 1e-8

base_pv = _to_float(oisbs_engine.value)
fd_sofr = (bumped_pv(ValuationEngineProductOISBasisSwap, oisbs_product, sofr_bump=EPS) - base_pv) / EPS
fd_flat = (bumped_pv(ValuationEngineProductOISBasisSwap, oisbs_product, flat_bump=EPS) - base_pv) / EPS
print('finite-diff (SOFR-1B)     :', fd_sofr, ' vs analytic:', grad[SLICES['SOFR-1B']].sum())
print('finite-diff (SOFR-1B-FLAT):', fd_flat, ' vs analytic:', grad[SLICES['SOFR-1B-FLAT']].sum())
assert abs(fd_sofr - grad[SLICES['SOFR-1B']].sum()) / abs(fd_sofr) < REL_TOL
assert abs(fd_flat - grad[SLICES['SOFR-1B-FLAT']].sum()) / abs(fd_flat) < REL_TOL
print('PASSED')

SOFR-1B block      sum: -5599.967640875373
USD-LIBOR-BBA-3M    sum: 0.0
SOFR-1B-FLAT block  sum: -5599.967640875631


finite-diff (SOFR-1B)     : -5599.965341389179  vs analytic: -5599.967640875373
finite-diff (SOFR-1B-FLAT): -5599.965341389179  vs analytic: -5599.967640875631
PASSED


## 9. `ValuationEngineProductOvernightIndexFuture`

A margined instrument -- `.value_ = notional*(F - K)`, undiscounted (a future is never present-valued), and `.cash_` is the daily variation margin `notional*(today's forward rate - yesterday's)`. Internally wraps `ValuationEngineProductOvernightIndexCompositeCashflow` (the future's own `product` object *is* a `ProductOvernightIndexCompositeCashflow`, `pay_or_rec=RECEIVE`, `spread=0`, `leverage=1`) and reads only its `forward_rate_`. `long=100` contracts, `contractual_notional=$1,000,000`, `basis_point=25` -> `notional = 100 * 1e6 * 25/1e4 = $250,000`.

In [51]:
fut_termination = add_period(EFFECTIVE, Period('3M'), on_biz_conv, on_hol_conv)
FUT_STRIKE = 0.05
fut_product = ProductOvernightIndexFuture(
    EFFECTIVE, TermOrDate(fut_termination), LongOrShort.LONG, 100, sofr_composite, FUT_STRIKE,
)
print('notional:', fut_product.notional, ' strike:', fut_product.strike)

notional: 250000.0  strike: 0.05


### 9a. `.value_` vs. closed form (fully forward)

In [52]:
fut_engine = ValuationEngineProductOvernightIndexFuture(yc, vpc, fut_product, ValuationRequest.PV)
fut_engine.calculate_value()

fut_df_eff = yc.discount_factor(sofr_index, fut_product.effective_date, calc_grad=True)
fut_df_term = yc.discount_factor(sofr_index, fut_product.termination_date, calc_grad=True)
fut_tau = fut_product.accrued
fut_expected_forward = (fut_df_eff / fut_df_term - 1.0) / fut_tau
fut_expected_value = fut_product.notional * (fut_expected_forward - fut_product.strike)

print('engine forward:', float(fut_engine.forward_rate_.detach()), ' closed:', float(fut_expected_forward.detach()))
print('engine value  :', _to_float(fut_engine.value), ' closed:', float(fut_expected_value.detach()))
assert abs(_to_float(fut_engine.value) - float(fut_expected_value.detach())) < 1e-6
print('PASSED')

engine forward: 0.042708817063390664  closed: 0.042708817063390664
engine value  : -1822.7957341523347  closed: -1822.7957341523347
PASSED


### 9b. `pv01()`/`grad_at_par()` -- exact closed forms (`.value_` is affine in `forward_rate_`, and `grad_at_par` differentiates `forward_rate_` itself, not scaled by notional)

In [53]:
fut_pv01 = fut_engine.pv01()
(fut_dv_df,) = torch.autograd.grad(fut_engine.value_, fut_engine.forward_rate_, retain_graph=True)
fut_expected_pv01 = float(fut_dv_df) * 1e-4
print('engine pv01:', fut_pv01, ' expected (d(value)/d(F) * 1bp, == notional * 1bp since affine):', fut_expected_pv01)
assert abs(fut_pv01 - fut_expected_pv01) < 1e-6
assert abs(fut_pv01 - fut_product.notional * 1e-4) < 1e-6

fut_gap = fut_engine.grad_at_par()
def bumped_fut_forward(**bumps):
    yc_b = build_model(VALUE_DATE, **bumps)
    e = ValuationEngineProductOvernightIndexFuture(yc_b, vpc, fut_product, ValuationRequest.PV)
    e.calculate_value()
    return float(e.forward_rate_.detach())

fut_base_forward = float(fut_engine.forward_rate_.detach())
fut_fd_forward_sofr = (bumped_fut_forward(sofr_bump=EPS) - fut_base_forward) / EPS
print('grad_at_par SOFR-1B analytic:', fut_gap[SLICES['SOFR-1B']].sum(), ' finite-diff of forward rate:', fut_fd_forward_sofr)
assert abs(fut_gap[SLICES['SOFR-1B']].sum() - fut_fd_forward_sofr) / abs(fut_fd_forward_sofr) < REL_TOL
print('PASSED')

engine pv01: 25.0  expected (d(value)/d(F) * 1bp, == notional * 1bp since affine): 25.0
grad_at_par SOFR-1B analytic: 0.9970663319721429  finite-diff of forward rate: 0.9970664580916111
PASSED


### 9c. Settlement/MTM branches: trade date (no mark yet), fully forward (rate is a pure discount-factor ratio between two fixed future dates, invariant to today's date, so cash is exactly zero even a day before the mark), mid-life (fixings partially realized -- genuine daily variation margin), and fully matured (`.value_` stays `notional*(F-K)`, *not* reset to zero -- a future's `.value_` is never a discounted PV, unlike every other engine in this file)

In [54]:
seed_daily_fixings(sofr_index, fut_product.effective_date, fut_product.termination_date, 0.0430, on_biz_conv, on_hol_conv)

# trade date -- nothing to mark against yet
yc_fut_trade = build_model(fut_product.effective_date)
fut_trade_engine = ValuationEngineProductOvernightIndexFuture(yc_fut_trade, vpc, fut_product, ValuationRequest.PV)
fut_trade_engine.calculate_value()
print('value_date == effective_date -- cash:', fut_trade_engine.cash)
assert fut_trade_engine.cash == 0.0

# mid-life -- partially realized fixings, genuine daily MTM
fut_mid_date = add_period(fut_product.effective_date, Period('1M'), on_biz_conv, on_hol_conv)
yc_fut_mid = build_model(fut_mid_date)
fut_mid_engine = ValuationEngineProductOvernightIndexFuture(yc_fut_mid, vpc, fut_product, ValuationRequest.PV)
fut_mid_engine.calculate_value()
print('mid-life -- value:', _to_float(fut_mid_engine.value), ' cash (variation margin):', fut_mid_engine.cash)
assert fut_mid_engine.cash != 0.0

# fully matured -- value_ stays notional*(F-K) (mark against final fixings), cash == 0 (nothing left to realize)
fut_matured_date = add_period(fut_product.payment_date, Period('1D'), on_biz_conv, on_hol_conv)
yc_fut_matured = build_model(fut_matured_date)
fut_matured_engine = ValuationEngineProductOvernightIndexFuture(yc_fut_matured, vpc, fut_product, ValuationRequest.PV)
fut_matured_engine.calculate_value()
print('fully matured -- value:', _to_float(fut_matured_engine.value), ' cash:', fut_matured_engine.cash)
assert fut_matured_engine.cash == 0.0
assert _to_float(fut_matured_engine.value) != 0.0  # NOT reset to zero, unlike a PV-based engine

# pv01 fallback once forward_rate_ has no live graph (fully realized)
print('matured pv01 (fallback == notional * 1bp):', fut_matured_engine.pv01(), ' expected:', fut_product.notional * 1e-4)
assert abs(fut_matured_engine.pv01() - fut_product.notional * 1e-4) < 1e-8
print('PASSED')

The fixings of SOFR-1B are all removed.
63 fixing(s) for SOFR-1B is(are) inserted.


value_date == effective_date -- cash: 0.0


mid-life -- value: -1785.0708627612319  cash (variation margin): 1.6111315924872542
fully matured -- value: -1692.0157166707902  cash: 0.0
matured pv01 (fallback == notional * 1bp): 25.0  expected: 25.0
PASSED


### 9d. `get_risk()` vs. finite difference

In [55]:
grad = np.zeros(N_STATE)
fut_engine.get_risk(gradient=grad)
print('SOFR-1B block      sum:', grad[SLICES['SOFR-1B']].sum())
print('USD-LIBOR-BBA-3M    sum:', grad[SLICES['USD-LIBOR-BBA-3M']].sum())
print('SOFR-1B-FLAT block  sum:', grad[SLICES['SOFR-1B-FLAT']].sum())
assert abs(grad[SLICES['USD-LIBOR-BBA-3M']].sum()) < 1e-8
assert abs(grad[SLICES['SOFR-1B-FLAT']].sum()) < 1e-8  # undiscounted -- no funding-curve dependency at all

base_value = _to_float(fut_engine.value)
fd_sofr = (bumped_pv(ValuationEngineProductOvernightIndexFuture, fut_product, sofr_bump=EPS) - base_value) / EPS
print('finite-diff (SOFR-1B):', fd_sofr, ' vs analytic:', grad[SLICES['SOFR-1B']].sum())
assert abs(fd_sofr - grad[SLICES['SOFR-1B']].sum()) / abs(fd_sofr) < REL_TOL
print('PASSED')

SOFR-1B block      sum: 249266.5829930357
USD-LIBOR-BBA-3M    sum: 0.0
SOFR-1B-FLAT block  sum: 0.0


finite-diff (SOFR-1B): 249266.61452286682  vs analytic: 249266.5829930357
PASSED


## 10. `ValuationEngineProductFRAOrFixing`

As of today's refactor this engine is IBOR-only: `self.index_: IBORIndex = product.index`, and it unconditionally wraps `ValuationEngineProductIBORIndexCashflow` (there is no more hand-built overnight branch). **Note on the actual current source** (`fixedincomelib/yield_curve/valuation_engine.py`, `ValuationEngineProductFRAOrFixing.__init__`/`.calculate_value`), read fresh for this test rather than assumed from a prior description: there is *no* explicit `isinstance(product.index, IBORIndex)` runtime guard anywhere in `__init__`, and only `fra_discounting_style.upper() == "ISDA"` is asserted. Passing an `OvernightIndex` in still *constructs* both the product and the engine without error -- it only fails inside `calculate_value()`, and even then via an unrelated internal `AssertionError` (an "irregular period" / index-family-tenor lookup deep in `ValuationEngineAnalyticsIborIndex`, not a deliberate type check at the FRA layer). 10e below demonstrates and documents this precisely as observed, rather than asserting a construction-time guard that doesn't actually exist.

Two payoff shapes selected purely by `payment_date_` vs. `termination_date_` (`pay_date_or_offset` is what controls which): a *plain fixing* (default `pay_date_or_offset=TermOrDate("0D")`, so `payment_date_ == termination_date_`) should price as `P(t,T)*N*(F-K)*tau`, and a *true, early-settling FRA* (`pay_date_or_offset` set to `effective_date`, so `payment_date_ == effective_date_ < termination_date_`) should price as `P(t,T)*N*(F-K)*tau/(1+F*tau)`.

In [56]:
fra_termination = add_period(EFFECTIVE, libor_3m.term, ib_biz_conv, ib_hol_conv)
FRA_COUPON = 0.047

fra_plain = ProductFRAOrFixing(
    EFFECTIVE, TermOrDate(fra_termination), PayOrReceive.RECEIVE, Currency('USD'), NOTIONAL,
    FRA_COUPON, libor_3m,
)
fra_true = ProductFRAOrFixing(
    EFFECTIVE, TermOrDate(fra_termination), PayOrReceive.RECEIVE, Currency('USD'), NOTIONAL,
    FRA_COUPON, libor_3m, pay_date_or_offset=TermOrDate(EFFECTIVE),
)
print('plain -- payment_date:', fra_plain.payment_date, ' termination_date:', fra_plain.termination_date)
print('true  -- payment_date:', fra_true.payment_date, ' termination_date:', fra_true.termination_date)
assert fra_plain.payment_date == fra_plain.termination_date
assert fra_true.payment_date < fra_true.termination_date

plain -- payment_date: November 19th, 2026  termination_date: November 19th, 2026
true  -- payment_date: August 19th, 2026  termination_date: November 19th, 2026


### 10a. PV vs. independent closed form (fully forward)

The true FRA's nonlinear closed form matches the engine exactly. **The plain fixing's PV does not match its documented closed form** -- see the discrepancy note right after.

In [57]:
fra_plain_engine = ValuationEngineProductFRAOrFixing(yc, vpc, fra_plain, ValuationRequest.PV)
fra_plain_engine.calculate_value()
fra_true_engine = ValuationEngineProductFRAOrFixing(yc, vpc, fra_true, ValuationRequest.PV)
fra_true_engine.calculate_value()

fra_tau = fra_plain.accrued
assert fra_tau == fra_true.accrued  # same effective/termination/index -> same day-count fraction
fra_df_pay_true = yc.discount_factor(funding_identifier, fra_true.payment_date, calc_grad=True)
F_true = float(fra_true_engine.forward_rate_.detach())
true_expected_pv = float(fra_df_pay_true.detach()) * NOTIONAL * (F_true - FRA_COUPON) * fra_tau / (1.0 + F_true * fra_tau)
print('true FRA  -- engine PV:', _to_float(fra_true_engine.value), ' closed (ISDA early-settlement):', true_expected_pv)
assert abs(_to_float(fra_true_engine.value) - true_expected_pv) < 1e-6
print('PASSED (true FRA)')

true FRA  -- engine PV: 101829.69291850147  closed (ISDA early-settlement): 101829.69291850145
PASSED (true FRA)


### 10a-note. Suspected library bug: the plain-fixing shape's forward-looking PV applies the ISDA early-settlement factor unconditionally

`ValuationEngineProductFRAOrFixing.calculate_value`'s forward-looking (`else`) branch computes `early_settlement_df = 1.0/(1.0 + self.forward_rate_*self.tau_)` and multiplies it into `self.value_` **unconditionally** -- it never branches on `self.payment_date_ < self.termination_date_` (`fixedincomelib/yield_curve/valuation_engine.py`, inside `ValuationEngineProductFRAOrFixing.calculate_value`, right where the comment literally reads "for a true (early-settling) FRA"). For the plain-fixing shape (`payment_date_ == termination_date_`), this factor should not apply at all -- the documented closed form (and the sibling `ValuationEngineProductIBORIndexCashflow` this engine wraps, which the plain-fixing case is structurally identical to) is `P(t,T)*N*(F-K)*tau`, no extra factor. This cell demonstrates the discrepancy via prints only (not a hard `assert`), so it does not block the rest of this notebook from running green; flagged in the session report instead of silently patching either the engine or the test oracle.

In [58]:
fra_df_pay_plain = yc.discount_factor(funding_identifier, fra_plain.payment_date, calc_grad=True)
F_plain = float(fra_plain_engine.forward_rate_.detach())
plain_documented_pv = float(fra_df_pay_plain.detach()) * NOTIONAL * (F_plain - FRA_COUPON) * fra_tau
print('plain fixing -- engine PV        :', _to_float(fra_plain_engine.value))
print('plain fixing -- documented closed:', plain_documented_pv, ' (P(t,T)*N*(F-K)*tau, no early-settlement factor)')
print('difference:', _to_float(fra_plain_engine.value) - plain_documented_pv)
print('NOT ASSERTED EQUAL -- see markdown note above; suspected bug, reported rather than patched.')

plain fixing -- engine PV        : 100730.2748048804
plain fixing -- documented closed: 102993.01728734803  (P(t,T)*N*(F-K)*tau, no early-settlement factor)
difference: -2262.7424824676273
NOT ASSERTED EQUAL -- see markdown note above; suspected bug, reported rather than patched.


### 10b. `par_rate_or_spread()` -- should be exactly `F` for both shapes, since `(F-K)` is a common factor in both formulas regardless of any extra multiplicative factor; PV at that coupon should be exactly 0 for both, and this check is unaffected by the 10a-note discrepancy

In [59]:
def fra_pv_at_coupon(product_ctor, coupon):
    p = product_ctor(coupon)
    e = ValuationEngineProductFRAOrFixing(yc, vpc, p, ValuationRequest.PV)
    e.calculate_value()
    return _to_float(e.value)

make_plain = lambda k: ProductFRAOrFixing(
    EFFECTIVE, TermOrDate(fra_termination), PayOrReceive.RECEIVE, Currency('USD'), NOTIONAL, k, libor_3m)
make_true = lambda k: ProductFRAOrFixing(
    EFFECTIVE, TermOrDate(fra_termination), PayOrReceive.RECEIVE, Currency('USD'), NOTIONAL, k, libor_3m,
    pay_date_or_offset=TermOrDate(EFFECTIVE))

plain_par = fra_plain_engine.par_rate_or_spread()
true_par = fra_true_engine.par_rate_or_spread()
print('plain par:', plain_par, ' true par:', true_par, ' (both should equal F)')
assert abs(plain_par - F_plain) < 1e-10
assert abs(true_par - F_true) < 1e-10
assert abs(fra_pv_at_coupon(make_plain, plain_par)) < 1e-6
assert abs(fra_pv_at_coupon(make_true, true_par)) < 1e-6
print('PASSED')

plain par: 0.08790018424108219  true par: 0.08790018424108219  (both should equal F)
PASSED


### 10c. `pv01()` -- the plain fixing's PV *as implemented* is not affine in F (per the 10a-note discrepancy), so this section only exercises the true FRA, whose PV genuinely is nonlinear in F by design (the ISDA early-settlement factor) -- verified against a finite difference of an independent closed form (not the engine's own formula), bumping F directly rather than the curve

In [60]:
def true_pv_closed(F):
    return float(fra_df_pay_true.detach()) * NOTIONAL * (F - FRA_COUPON) * fra_tau / (1.0 + F * fra_tau)

fd_eps = 1e-6
fd_true_pv01 = (true_pv_closed(F_true + fd_eps) - true_pv_closed(F_true - fd_eps)) / (2 * fd_eps) * 1e-4
print('true FRA pv01 -- engine:', fra_true_engine.pv01(), ' finite-diff of independent closed form:', fd_true_pv01)
assert abs(fra_true_engine.pv01() - fd_true_pv01) < 1e-3
print('PASSED')

true FRA pv01 -- engine: 246.4260909060162  finite-diff of independent closed form: 246.4260909058794
PASSED


### 10d. Settlement-date branches

`value_date > payment_date` (fully matured -- unaffected by the 10a-note discrepancy, works for both shapes) and `value_date == payment_date` (needs the period's own fixing seeded, since `fixing_date <= value_date` at that point). The plain fixing's on-payment-date value is correct (matches its documented closed form exactly, since the `elif` branch never applies the early-settlement factor at all -- see 10d-note for the mirror-image discrepancy this creates for the true FRA).

In [61]:
set_single_fixing(libor_3m, fra_plain.fixing_date, 0.0455)
assert fra_plain.fixing_date == fra_true.fixing_date  # same effective_date -> same IBOR fixing lag

# matured -- both shapes
fra_plain_matured_date = add_period(fra_plain.payment_date, Period('1D'), ib_biz_conv, ib_hol_conv)
fra_true_matured_date = add_period(fra_true.payment_date, Period('1D'), ib_biz_conv, ib_hol_conv)
for label, product, matured_date in [('plain', fra_plain, fra_plain_matured_date), ('true', fra_true, fra_true_matured_date)]:
    yc_m = build_model(matured_date)
    e = ValuationEngineProductFRAOrFixing(yc_m, vpc, product, ValuationRequest.PV)
    e.calculate_value()
    print(f'{label} matured -- value: {e.value}  cash: {e.cash}')
    assert e.value == 0.0 and e.cash == 0.0

# on payment date -- plain fixing (correct, no early-settlement factor expected or applied)
yc_plain_pay = build_model(fra_plain.payment_date)
fra_plain_pay_engine = ValuationEngineProductFRAOrFixing(yc_plain_pay, vpc, fra_plain, ValuationRequest.PV)
fra_plain_pay_engine.calculate_value()
expected_on_pay = NOTIONAL * fra_tau * (0.0455 - FRA_COUPON)
print('plain on payment date -- engine value:', fra_plain_pay_engine.value, ' expected:', expected_on_pay)
assert fra_plain_pay_engine.df_ == 1.0 and fra_plain_pay_engine.value == fra_plain_pay_engine.cash
assert abs(fra_plain_pay_engine.value - expected_on_pay) < 1e-6
assert fra_plain_pay_engine.pv01() == 0.0  # no live graph left (rate came from a historical fixing)
print('PASSED (plain fixing branches)')

The fixings of USD-LIBOR-BBA-3M are all removed.
1 fixing(s) for USD-LIBOR-BBA-3M is(are) inserted.
plain matured -- value: 0.0  cash: 0.0
true matured -- value: 0.0  cash: 0.0
plain on payment date -- engine value: -3833.3333333333367  expected: -3833.3333333333367
PASSED (plain fixing branches)


### 10d-note. Suspected library bug (mirror image of 10a-note): the true FRA's on-payment-date branch never applies the ISDA early-settlement factor

`ValuationEngineProductFRAOrFixing.calculate_value`'s `elif self.value_date_ == self.payment_date_:` branch sets `self.value_ = settlement_amount` directly, with no `early_settlement_df` factor at all, in *either* shape. For the plain fixing this is correct (10d above). For the true, early-settling FRA it is not: the ISDA cash-settlement amount paid *on* the settlement date is inherently `N*(F-K)*tau/(1+F*tau)`, not the undiscounted `N*(F-K)*tau` -- the `1/(1+F*tau)` factor is part of the definition of the settlement amount itself, not a "discount from today to payment_date" adjustment that should vanish just because `value_date_` and `payment_date_` coincide. Demonstrated below via prints only, not a hard `assert`, for the same reason as 10a-note.

In [62]:
yc_true_pay = build_model(fra_true.payment_date)
fra_true_pay_engine = ValuationEngineProductFRAOrFixing(yc_true_pay, vpc, fra_true, ValuationRequest.PV)
fra_true_pay_engine.calculate_value()
F_on_pay = _to_float(fra_true_pay_engine.forward_rate_)
raw_settlement = NOTIONAL * fra_tau * (F_on_pay - FRA_COUPON)
isda_settlement = raw_settlement / (1.0 + F_on_pay * fra_tau)
print('true FRA on payment date -- engine value (raw, no factor)     :', fra_true_pay_engine.value)
print('true FRA on payment date -- ISDA-adjusted (documented-correct):', isda_settlement)
print('difference:', fra_true_pay_engine.value - isda_settlement)
print('NOT ASSERTED EQUAL -- see markdown note above; suspected bug, reported rather than patched.')

true FRA on payment date -- engine value (raw, no factor)     : -3833.3333333333367
true FRA on payment date -- ISDA-adjusted (documented-correct): -3789.272514594194
difference: -44.06081873914263
NOT ASSERTED EQUAL -- see markdown note above; suspected bug, reported rather than patched.


### 10e. `get_risk()` vs. finite difference (true FRA, fully forward) -- unaffected by either discrepancy above, since this checks internal consistency (autograd vs. finite difference of the *same*, however-defined, PV formula), not the formula's absolute correctness

In [63]:
fra_true_engine2 = ValuationEngineProductFRAOrFixing(yc, vpc, fra_true, ValuationRequest.PV)
fra_true_engine2.calculate_value()
grad = np.zeros(N_STATE)
fra_true_engine2.get_risk(gradient=grad)
print('SOFR-1B block      sum:', grad[SLICES['SOFR-1B']].sum())
print('USD-LIBOR-BBA-3M    sum:', grad[SLICES['USD-LIBOR-BBA-3M']].sum())
print('SOFR-1B-FLAT block  sum:', grad[SLICES['SOFR-1B-FLAT']].sum())

base_pv = _to_float(fra_true_engine2.value)
fd_sofr = (bumped_pv(ValuationEngineProductFRAOrFixing, fra_true, sofr_bump=EPS) - base_pv) / EPS
fd_libor = (bumped_pv(ValuationEngineProductFRAOrFixing, fra_true, libor_bump=EPS) - base_pv) / EPS
fd_flat = (bumped_pv(ValuationEngineProductFRAOrFixing, fra_true, flat_bump=EPS) - base_pv) / EPS
print('finite-diff (SOFR-1B)         :', fd_sofr, ' vs analytic:', grad[SLICES['SOFR-1B']].sum())
print('finite-diff (USD-LIBOR-BBA-3M):', fd_libor, ' vs analytic:', grad[SLICES['USD-LIBOR-BBA-3M']].sum())
print('finite-diff (SOFR-1B-FLAT)    :', fd_flat, ' vs analytic:', grad[SLICES['SOFR-1B-FLAT']].sum())
assert abs(fd_sofr - grad[SLICES['SOFR-1B']].sum()) / abs(fd_sofr) < REL_TOL
assert abs(fd_libor - grad[SLICES['USD-LIBOR-BBA-3M']].sum()) / abs(fd_libor) < REL_TOL
assert abs(fd_flat - grad[SLICES['SOFR-1B-FLAT']].sum()) / abs(fd_flat) < REL_TOL
print('PASSED')

SOFR-1B block      sum: 2475894.7240668507
USD-LIBOR-BBA-3M    sum: 2485101.244248524
SOFR-1B-FLAT block  sum: -9206.520181672731
finite-diff (SOFR-1B)         : 2475894.185801735  vs analytic: 2475894.7240668507
finite-diff (USD-LIBOR-BBA-3M): 2485100.932404748  vs analytic: 2485101.244248524
finite-diff (SOFR-1B-FLAT)    : -9206.519782310352  vs analytic: -9206.520181672731
PASSED


### 10f. Constructing with an overnight index: current actual behavior (documented above -- succeeds at construction, fails only inside `calculate_value()`, via an unrelated internal assertion)

In [64]:
fra_on_termination = add_period(EFFECTIVE, Period('3M'), ib_biz_conv, ib_hol_conv)
fra_on_product = ProductFRAOrFixing(
    EFFECTIVE, TermOrDate(fra_on_termination), PayOrReceive.RECEIVE, Currency('USD'), NOTIONAL,
    FRA_COUPON, sofr_index,
)
print('construction with an OvernightIndex succeeded (no type guard in __init__)')
fra_on_engine = ValuationEngineProductFRAOrFixing(yc, vpc, fra_on_product, ValuationRequest.PV)
print('engine construction also succeeded')

raised = None
try:
    fra_on_engine.calculate_value()
except AssertionError as e:
    raised = e
print('calculate_value() raised:', type(raised), raised)
assert raised is not None
print('PASSED (documents actual behavior; not the construction-time type guard one might expect)')

construction with an OvernightIndex succeeded (no type guard in __init__)
engine construction also succeeded
calculate_value() raised: <class 'AssertionError'> model has no calibrated tenor for the index family of SOFR-1B
PASSED (documents actual behavior; not the construction-time type guard one might expect)


## 11. `ValuationEngineProductCashDeposit`

No index at all (`ProductCashDeposit` is a pure fixed cashflow) -- the only curve dependency is the funding/discount curve directly. `PV = sign*(notional*(1+coupon*tau)*DF(payment_date) - notional*DF(effective_date))`. The principal/funding leg (the second term) is only live while `value_date_ < effective_date_` (forward-starting); once the deposit has actually started, that principal exchange is sunk and drops out of a forward PV. `coupon_` is a plain Python float, never wrapped in a tensor, so `pv01()` is a closed form rather than an autograd call.

In [65]:
cd_termination = add_period(EFFECTIVE, Period('3M'), on_biz_conv, on_hol_conv)
CD_COUPON = 0.044
cd_product = ProductCashDeposit(
    EFFECTIVE, TermOrDate(cd_termination), PayOrReceive.RECEIVE, Currency('USD'), NOTIONAL,
    CD_COUPON, AccrualBasis.new('ACTUAL/360'),
    pay_business_day_convention=on_biz_conv, pay_holiday_convention=on_hol_conv,
)
print('effective:', cd_product.effective_date, ' payment:', cd_product.payment_date, ' value_date:', yc.value_date)
assert yc.value_date < cd_product.effective_date  # forward-starting against the shared curve

effective: August 19th, 2026  payment: November 19th, 2026  value_date: July 17th, 2026


### 11a. PV vs. closed form -- forward-starting deposit (principal leg live)

In [66]:
cd_engine = ValuationEngineProductCashDeposit(yc, vpc, cd_product, ValuationRequest.PV)
cd_engine.calculate_value()

cd_tau = cd_product.accrued
cd_df_pay = yc.discount_factor(funding_identifier, cd_product.payment_date, calc_grad=True)
cd_df_eff = yc.discount_factor(funding_identifier, cd_product.effective_date, calc_grad=True)
cd_expected_pv = (
    NOTIONAL * (1.0 + cd_tau * CD_COUPON) * float(cd_df_pay.detach())
    - NOTIONAL * float(cd_df_eff.detach())
)
print('engine PV:', _to_float(cd_engine.value), ' closed:', cd_expected_pv)
assert abs(_to_float(cd_engine.value) - cd_expected_pv) < 1e-6
print('PASSED')

engine PV: 3251.399204645306  closed: 3251.399204645306
PASSED


### 11b. Already-started deposit -- `effective_date_` before the curve's `value_date` -- the principal leg is sunk and drops out; PV should match the interest-only closed form

In [67]:
cd_started_effective = subtract_period(yc.value_date, Period('2M'), on_biz_conv, on_hol_conv)
cd_started_termination = add_period(cd_started_effective, Period('3M'), on_biz_conv, on_hol_conv)
cd_started_product = ProductCashDeposit(
    cd_started_effective, TermOrDate(cd_started_termination), PayOrReceive.RECEIVE, Currency('USD'), NOTIONAL,
    CD_COUPON, AccrualBasis.new('ACTUAL/360'),
    pay_business_day_convention=on_biz_conv, pay_holiday_convention=on_hol_conv,
)
print('started deposit -- effective:', cd_started_product.effective_date, ' payment:', cd_started_product.payment_date, ' value_date:', yc.value_date)
assert cd_started_product.effective_date < yc.value_date < cd_started_product.payment_date

cd_started_engine = ValuationEngineProductCashDeposit(yc, vpc, cd_started_product, ValuationRequest.PV)
cd_started_engine.calculate_value()

cd_started_tau = cd_started_product.accrued
cd_started_df_pay = yc.discount_factor(funding_identifier, cd_started_product.payment_date, calc_grad=True)
cd_started_expected_pv = NOTIONAL * (1.0 + cd_started_tau * CD_COUPON) * float(cd_started_df_pay.detach())
print('engine PV:', _to_float(cd_started_engine.value), ' closed (interest leg only, no principal leg):', cd_started_expected_pv)
assert abs(_to_float(cd_started_engine.value) - cd_started_expected_pv) < 1e-6
print('PASSED')

started deposit -- effective: May 18th, 2026  payment: August 18th, 2026  value_date: July 17th, 2026
engine PV: 10074393.68230003  closed (interest leg only, no principal leg): 10074393.68230003
PASSED


### 11c. `par_rate_or_spread()` (curve-implied simple rate that zeros the two-leg PV) and `pv01()` -- both exact closed forms

In [68]:
cd_par = cd_engine.par_rate_or_spread()
cd_par_expected = (float(cd_df_eff.detach()) / float(cd_df_pay.detach()) - 1.0) / cd_tau
print('engine par:', cd_par, ' closed:', cd_par_expected)
assert abs(cd_par - cd_par_expected) < 1e-10

cd_product_at_par = ProductCashDeposit(
    EFFECTIVE, TermOrDate(cd_termination), PayOrReceive.RECEIVE, Currency('USD'), NOTIONAL,
    cd_par, AccrualBasis.new('ACTUAL/360'),
    pay_business_day_convention=on_biz_conv, pay_holiday_convention=on_hol_conv,
)
cd_engine_at_par = ValuationEngineProductCashDeposit(yc, vpc, cd_product_at_par, ValuationRequest.PV)
cd_engine_at_par.calculate_value()
print('PV at par coupon (should be ~0):', _to_float(cd_engine_at_par.value))
assert abs(_to_float(cd_engine_at_par.value)) < 1e-2

cd_pv01 = cd_engine.pv01()
cd_pv01_expected = NOTIONAL * cd_tau * float(cd_df_pay.detach()) * 1e-4
print('engine pv01:', cd_pv01, ' closed:', cd_pv01_expected)
assert abs(cd_pv01 - cd_pv01_expected) < 1e-6
print('PASSED')

engine par: 0.042708817063390664  closed: 0.042708817063390664
PV at par coupon (should be ~0): 0.0
engine pv01: 251.8155338378567  closed: 251.8155338378567
PASSED


### 11d. `get_risk()` vs. finite difference

In [69]:
grad = np.zeros(N_STATE)
cd_engine.get_risk(gradient=grad)
print('SOFR-1B block      sum:', grad[SLICES['SOFR-1B']].sum())
print('USD-LIBOR-BBA-3M    sum:', grad[SLICES['USD-LIBOR-BBA-3M']].sum())
print('SOFR-1B-FLAT block  sum:', grad[SLICES['SOFR-1B-FLAT']].sum())
assert abs(grad[SLICES['USD-LIBOR-BBA-3M']].sum()) < 1e-8

base_pv = _to_float(cd_engine.value)
fd_sofr = (bumped_pv(ValuationEngineProductCashDeposit, cd_product, sofr_bump=EPS) - base_pv) / EPS
fd_flat = (bumped_pv(ValuationEngineProductCashDeposit, cd_product, flat_bump=EPS) - base_pv) / EPS
print('finite-diff (SOFR-1B)     :', fd_sofr, ' vs analytic:', grad[SLICES['SOFR-1B']].sum())
print('finite-diff (SOFR-1B-FLAT):', fd_flat, ' vs analytic:', grad[SLICES['SOFR-1B-FLAT']].sum())
assert abs(fd_sofr - grad[SLICES['SOFR-1B']].sum()) / abs(fd_sofr) < REL_TOL
assert abs(fd_flat - grad[SLICES['SOFR-1B-FLAT']].sum()) / abs(fd_flat) < REL_TOL
print('PASSED')

SOFR-1B block      sum: -2511881.3994514905
USD-LIBOR-BBA-3M    sum: 0.0
SOFR-1B-FLAT block  sum: -2511881.3994514905


finite-diff (SOFR-1B)     : -2511880.8560073376  vs analytic: -2511881.3994514905
finite-diff (SOFR-1B-FLAT): -2511880.8560073376  vs analytic: -2511881.3994514905
PASSED


## Additional setup for sections 12-14

Imports for the three `ARTIFICIAL PRODUCT` engines at the bottom of `valuation_engine.py`: `ValuationEngineProductGenericForward`, `ValuationEngineProductGenericSpread`, and `ValuationEngineProductGenericForwardSpread` (the last subclasses the second). No new curve components are needed -- all three sections reuse the shared `yc`/`vpc`/`SLICES`/`N_STATE` fixture built at the top of this notebook.

In [70]:
from fixedincomelib.yield_curve.valuation_engine import (
    ValuationEngineProductGenericForward,
    ValuationEngineProductGenericSpread,
    ValuationEngineProductGenericForwardSpread,
)
from fixedincomelib.product.linear_products import (
    ProductGenericForward,
    ProductGenericSpread,
    ProductGenericForwardSpread,
)

print('Additional imports for sections 12-14 loaded.')


Additional imports for sections 12-14 loaded.


## 12. `ValuationEngineProductGenericForward`

An implied-forward instrument: `PV = sign*notional*(F-K)*tau*DF_funding(payment_date)`, where `F` is the curve-implied forward rate of `product.index` over `[effective_date, termination_date]`, read directly off the ratio of that index's own discount factors (`model.discount_factor(product.index, date, calc_grad=True)`), *not* through an `AnchoredIborIndex`/`ValuationEngineAnalytics*` pair the way every engine above is. `product.index` here is `USD-LIBOR-BBA-3M` (the plain `IBORIndex`, same one section 3 uses), 9M span, 4.6% strike. Both `SIMPLE` and `CONTINUOUS` compounding are checked -- they must give different `F` (and PV) but *identical* `pv01()`, since `pv01()` is `d(PV)/dF`, which doesn't care how `F` itself was derived.

In [71]:
gf_termination = add_period(EFFECTIVE, Period('9M'), ib_biz_conv, ib_hol_conv)
GF_COUPON = 0.046
gf_accrual_basis = AccrualBasis.new('ACTUAL/360')

gf_products = {}
gf_engines = {}
for method in [CompoundingMethod.SIMPLE, CompoundingMethod.CONTINUOUS]:
    gf_products[method] = ProductGenericForward(
        EFFECTIVE, TermOrDate(gf_termination), PayOrReceive.RECEIVE, Currency('USD'), NOTIONAL,
        GF_COUPON, libor_3m, accrual_basis=gf_accrual_basis,
        business_day_convention=ib_biz_conv, holiday_convention=ib_hol_conv,
        payment_business_day_conv=ib_biz_conv, payment_holiday_conv=ib_hol_conv,
        compounding_method=method,
    )
    gf_engines[method] = ValuationEngineProductGenericForward(yc, vpc, gf_products[method], ValuationRequest.PV)
    gf_engines[method].calculate_value()

print('effective:', EFFECTIVE, ' termination:', gf_termination, ' pay_date:', gf_products[CompoundingMethod.SIMPLE].pay_date)


effective: August 19th, 2026  termination: May 19th, 2027  pay_date: May 19th, 2027


### 12a. PV vs. closed form, both compounding methods

In [72]:
gf_tau = accrued(EFFECTIVE, gf_termination, gf_accrual_basis, ib_biz_conv, ib_hol_conv)
gf_df_eff = yc.discount_factor(libor_3m, EFFECTIVE, calc_grad=True)
gf_df_term = yc.discount_factor(libor_3m, gf_termination, calc_grad=True)
gf_growth = gf_df_eff / gf_df_term
gf_df_pay = yc.discount_factor(funding_identifier, gf_termination, calc_grad=True)

for method, engine in gf_engines.items():
    F_expected = torch.log(gf_growth) / gf_tau if method == CompoundingMethod.CONTINUOUS else (gf_growth - 1.0) / gf_tau
    pv_expected = NOTIONAL * gf_tau * (F_expected - GF_COUPON) * gf_df_pay
    print(f'--- {method} ---')
    print('engine forward:', float(engine.forward_rate_.detach()), ' closed:', float(F_expected.detach()))
    print('engine PV     :', _to_float(engine.value), ' closed:', float(pv_expected.detach()))
    assert abs(float(engine.forward_rate_.detach()) - float(F_expected.detach())) < 1e-10
    assert abs(_to_float(engine.value) - float(pv_expected.detach())) < 1e-6
print('PASSED')


--- CompoundingMethod.SIMPLE ---
engine forward: 0.09032092499762352  closed: 0.09032092499762352
engine PV     : 324130.321005184  closed: 324130.321005184
--- CompoundingMethod.CONTINUOUS ---
engine forward: 0.0873620954388077  closed: 0.0873620954388077
engine PV     : 302491.6396204877  closed: 302491.6396204877
PASSED


### 12b. `pv01()` exact closed form -- identical across both compounding methods, since it's `d(PV)/dF` and doesn't depend on how `F` itself was derived

In [73]:
gf_pv01_expected = NOTIONAL * gf_tau * float(gf_df_pay.detach()) * 1e-4
for method, engine in gf_engines.items():
    pv01 = engine.pv01()
    print(f'{method} pv01:', pv01, ' expected:', gf_pv01_expected)
    assert abs(pv01 - gf_pv01_expected) < 1e-6
print('PASSED')


CompoundingMethod.SIMPLE pv01: 731.3257135823853  expected: 731.3257135823853
CompoundingMethod.CONTINUOUS pv01: 731.3257135823853  expected: 731.3257135823853
PASSED


### 12c. Settlement-date branches: on pay date (cash realized) and fully matured

In [74]:
gf_product = gf_products[CompoundingMethod.SIMPLE]

yc_gf_pay = build_model(gf_product.pay_date)
gf_pay_engine = ValuationEngineProductGenericForward(yc_gf_pay, vpc, gf_product, ValuationRequest.PV)
gf_pay_engine.calculate_value()
print('value_date == pay_date -> value:', gf_pay_engine.value, ' cash:', gf_pay_engine.cash, ' df:', gf_pay_engine.df_)
assert gf_pay_engine.df_ == 1.0 and gf_pay_engine.value == gf_pay_engine.cash and gf_pay_engine.cash != 0.0

gf_matured_date = add_period(gf_product.pay_date, Period('1D'), ib_biz_conv, ib_hol_conv)
yc_gf_matured = build_model(gf_matured_date)
gf_matured_engine = ValuationEngineProductGenericForward(yc_gf_matured, vpc, gf_product, ValuationRequest.PV)
gf_matured_engine.calculate_value()
print('value_date >  pay_date -> value:', gf_matured_engine.value, ' cash:', gf_matured_engine.cash)
assert gf_matured_engine.value == 0.0 and gf_matured_engine.cash == 0.0
print('PASSED')


value_date == pay_date -> value: tensor(-348833.3333, dtype=torch.float64, grad_fn=<MulBackward0>)  cash: -348833.3333333333  df: 1.0
value_date >  pay_date -> value: 0.0  cash: 0.0
PASSED


### 12d. `get_risk()` vs. finite difference -- nonzero on all three components (`SOFR-1B` indirectly via `USD-LIBOR-BBA-3M`/`SOFR-1B-FLAT` both being `REFERENCE`d off it, `USD-LIBOR-BBA-3M` directly via `product.index`, `SOFR-1B-FLAT` directly via discounting), same shape as section 3d

In [75]:
gf_engine = gf_engines[CompoundingMethod.SIMPLE]
grad = np.zeros(N_STATE)
gf_engine.get_risk(gradient=grad)
print('SOFR-1B block      sum:', grad[SLICES['SOFR-1B']].sum())
print('USD-LIBOR-BBA-3M    sum:', grad[SLICES['USD-LIBOR-BBA-3M']].sum())
print('SOFR-1B-FLAT block  sum:', grad[SLICES['SOFR-1B-FLAT']].sum())

base_pv = _to_float(gf_engine.value)
fd_sofr = (bumped_pv(ValuationEngineProductGenericForward, gf_product, sofr_bump=EPS) - base_pv) / EPS
fd_libor = (bumped_pv(ValuationEngineProductGenericForward, gf_product, libor_bump=EPS) - base_pv) / EPS
fd_flat = (bumped_pv(ValuationEngineProductGenericForward, gf_product, flat_bump=EPS) - base_pv) / EPS
print('finite-diff (SOFR-1B)         :', fd_sofr, ' vs analytic:', grad[SLICES['SOFR-1B']].sum())
print('finite-diff (USD-LIBOR-BBA-3M):', fd_libor, ' vs analytic:', grad[SLICES['USD-LIBOR-BBA-3M']].sum())
print('finite-diff (SOFR-1B-FLAT)    :', fd_flat, ' vs analytic:', grad[SLICES['SOFR-1B-FLAT']].sum())
assert abs(fd_sofr - grad[SLICES['SOFR-1B']].sum()) / abs(fd_sofr) < REL_TOL
assert abs(fd_libor - grad[SLICES['USD-LIBOR-BBA-3M']].sum()) / abs(fd_libor) < REL_TOL
assert abs(fd_flat - grad[SLICES['SOFR-1B-FLAT']].sum()) / abs(fd_flat) < REL_TOL
print('PASSED')


SOFR-1B block      sum: 7435386.716205731
USD-LIBOR-BBA-3M    sum: 7707123.368884049
SOFR-1B-FLAT block  sum: -271736.6526783187
finite-diff (SOFR-1B)         : 7435383.250354789  vs analytic: 7435386.716205731
finite-diff (USD-LIBOR-BBA-3M): 7707126.252586022  vs analytic: 7707123.368884049
finite-diff (SOFR-1B-FLAT)    : -271736.53879435733  vs analytic: -271736.6526783187
PASSED


## 13. `ValuationEngineProductGenericSpread`

Prices a spread between a "basis"/target instrument T (`product.basis_data_convention`, built at coupon `0.`) and a "reference" instrument R (`product.reference_data_convention`, built at coupon `-product.spread`), both constructed via `ProductFactory.create_product_from_data_convention` and dispatched through `ValuationEngineProductRegistry` -- so T and R can be *any* two data-convention-driven instrument types (a swap vs. a deposit, an FRA vs. an OIS, ...), unlike every engine above, which combines legs that are already the same atomic-cashflow family.

**Known blocking issue (not fixed here, per this session's scope):** `ValuationEngineProductGenericSpread.create_cash_flows_report()` is currently commented out in `fixedincomelib/yield_curve/valuation_engine.py`, leaving the abstract method inherited from `ValuationEngineProduct` unimplemented -- so this class is **not instantiable** at all right now. The cell below demonstrates that precisely, with a realistic `ProductGenericSpread` (an `OVERNIGHT INDEX SWAP` basis leg vs. a `CASH DEPOSIT` reference leg, both registered as ad hoc data conventions here), and stops there -- there is no way to exercise this class's own `calculate_value`/`par_rate_or_spread`/`pv01` as a live object right now. That shared logic *is* still exercised indirectly in section 14 below, via the subclass `ValuationEngineProductGenericForwardSpread` (which supplies its own concrete `create_cash_flows_report` override and is instantiable) -- but only for `ProductGenericForward` legs specifically (see 14b), not for the coupon-style legs (swap/deposit) this section's fixture uses, so this session's testing does not confirm the base class's own combination logic is correct when T/R are coupon-style instruments -- only that it is currently unreachable to test at all.

In [76]:
DataConventionRegistry().register("USD-SOFR-OIS-GENERIC", {
    "type": "OVERNIGHT INDEX SWAP",
    "convention": {
        "SETTLEMENT OFFSET": "2D",
        "SETTLEMENT HOLIDAY CONVENTION": "NYC",
        "CURRENCY": "USD",
        "NOTIONAL": "10000000",
        "ACCRUAL PERIOD": "3M",
        "ACCRUAL BASIS": "ACTUAL/360",
        "INDEX": "USD-SOFR-COMPOUND",
        "RATE CUTOFF": "0D",
        "PAYMENT OFFSET": "0D",
        "PAYMENT BUSINESS DAY CONVENTION": "F",
        "PAYMENT HOLIDAY CONVENTION": "NYC",
    },
})
DataConventionRegistry().register("USD-CASH-DEPOSIT-GENERIC", {
    "type": "CASH DEPOSIT",
    "convention": {
        "SETTLEMENT OFFSET": "2D",
        "SETTLEMENT HOLIDAY CONVENTION": "NYC",
        "CURRENCY": "USD",
        "NOTIONAL": "10000000",
        "ACCRUAL BASIS": "ACTUAL/360",
        "PAYMENT BUSINESS DAY CONVENTION": "F",
        "PAYMENT HOLIDAY CONVENTION": "NYC",
        "END OF MONTH": "",
    },
})

basis_dc = DataConventionRegistry().get("USD-SOFR-OIS-GENERIC")
reference_dc = DataConventionRegistry().get("USD-CASH-DEPOSIT-GENERIC")

GS_SPREAD = 0.0009
gs_product = ProductGenericSpread(
    EFFECTIVE, TermOrDate('2Y'), PayOrReceive.RECEIVE, Currency('USD'), NOTIONAL, GS_SPREAD,
    basis_dc, reference_dc,
)

raised = None
try:
    gs_engine = ValuationEngineProductGenericSpread(yc, vpc, gs_product, ValuationRequest.PV)
except TypeError as e:
    raised = e
print('construction raised:', type(raised), raised)
assert raised is not None
assert 'create_cash_flows_report' in str(raised)
print('PASSED (blocked-state confirmed, matches known issue -- not worked around)')


construction raised: <class 'TypeError'> Can't instantiate abstract class ValuationEngineProductGenericSpread with abstract method create_cash_flows_report
PASSED (blocked-state confirmed, matches known issue -- not worked around)


## 14. `ValuationEngineProductGenericForwardSpread`

Subclasses `ValuationEngineProductGenericSpread`, spreading two *implied forward rates* over the same period: `basis_index` (here `USD-LIBOR-BBA-3M`) vs. `reference_index` (here `SOFR-1B`, the plain overnight index, not the composite). It overrides `__init__` (building two `ProductGenericForward` legs, both on `product.notional` so `scale_` is always `1.0`, each in its own index's native currency -- same currency here, `USD`, so `fx_` is trivially `1.0`) and `create_cash_flows_report`, but inherits `calculate_value`/`par_rate_or_spread`/`pv01` unchanged from the base class -- confirmed empirically below to be instantiable despite the parent's `create_cash_flows_report` being commented out (Python's ABC machinery only requires the *most-derived* class to supply a concrete override, which this subclass does).

In [77]:
GFS_TERM = TermOrDate('1Y')
GFS_SPREAD = 0.0011
gfs_accrual_basis = AccrualBasis.new('ACTUAL/360')

gfs_product = ProductGenericForwardSpread(
    EFFECTIVE, GFS_TERM, PayOrReceive.RECEIVE, Currency('USD'), NOTIONAL, GFS_SPREAD,
    basis_index=libor_3m, reference_index=sofr_index,
    accrual_basis=gfs_accrual_basis,
    business_day_convention=ib_biz_conv, holiday_convention=ib_hol_conv,
    payment_business_day_conv=ib_biz_conv, payment_holiday_conv=ib_hol_conv,
    compounding_method=CompoundingMethod.SIMPLE,
)

gfs_engine = ValuationEngineProductGenericForwardSpread(yc, vpc, gfs_product, ValuationRequest.PV)
print('Instantiated OK (subclass supplies its own concrete create_cash_flows_report).')
gfs_engine.calculate_value()
print('target_engine_ type   :', type(gfs_engine.target_engine_).__name__)
print('reference_engine_ type:', type(gfs_engine.reference_engine_).__name__)
assert isinstance(gfs_engine.target_engine_, ValuationEngineProductGenericForward)
assert isinstance(gfs_engine.reference_engine_, ValuationEngineProductGenericForward)
print('scale_:', gfs_engine.scale_, ' fx_:', gfs_engine.fx_)
assert gfs_engine.scale_ == 1.0  # both legs built on product.notional


Instantiated OK (subclass supplies its own concrete create_cash_flows_report).
target_engine_ type   : ValuationEngineProductGenericForward
reference_engine_ type: ValuationEngineProductGenericForward
scale_: 1.0  fx_: 1.0


### 14a. `.value_` vs. an independent closed form

The combination formula is `value_ = sign*scale*fx*(R.value - ratio*T.value)`, `ratio = R.pv01()/T.pv01()`. `scale_`/`fx_` are both `1.0` here (see header above). T and R share the same dates, day-count basis, notional and funding discounting, and both are always built `RECEIVE` -- so `T.pv01() == R.pv01()` exactly here, making `ratio_` come out to exactly `1.0` too (checked below, not assumed).

In [78]:
gfs_termination = gfs_product.termination_date
gfs_tau = accrued(EFFECTIVE, gfs_termination, gfs_accrual_basis, ib_biz_conv, ib_hol_conv)
gfs_df_eff_T = yc.discount_factor(libor_3m, EFFECTIVE, calc_grad=True)
gfs_df_term_T = yc.discount_factor(libor_3m, gfs_termination, calc_grad=True)
gfs_F_T = (gfs_df_eff_T / gfs_df_term_T - 1.0) / gfs_tau
gfs_df_eff_R = yc.discount_factor(sofr_index, EFFECTIVE, calc_grad=True)
gfs_df_term_R = yc.discount_factor(sofr_index, gfs_termination, calc_grad=True)
gfs_F_R = (gfs_df_eff_R / gfs_df_term_R - 1.0) / gfs_tau
gfs_df_pay = yc.discount_factor(funding_identifier, gfs_termination, calc_grad=True)

# value_ = sign*(N*tau*df*(F_R+spread) - ratio*N*tau*df*F_T), ratio == 1.0 here
gfs_expected_value = NOTIONAL * gfs_tau * float(gfs_df_pay.detach()) * (
    float(gfs_F_R.detach()) + GFS_SPREAD - float(gfs_F_T.detach())
)
print('engine value_        :', _to_float(gfs_engine.value))
print('independent closed form:', gfs_expected_value)
assert abs(_to_float(gfs_engine.value) - gfs_expected_value) < 1e-4

print('ratio_ (expected exactly 1.0):', gfs_engine.ratio_)
assert abs(gfs_engine.ratio_ - 1.0) < 1e-8
print('PASSED')


engine value_        : -451680.8327576997
independent closed form: -451680.83275769965
ratio_ (expected exactly 1.0): 1.0
PASSED


### 14b. Suspected library bug: `pv01()`/`par_rate_or_spread()` have the wrong sign for this class

`fixedincomelib/yield_curve/valuation_engine.py`, `ValuationEngineProductGenericSpread.calculate_value`, the line `self.spread_pv01_unit_ = -self.sign_ * self.scale_ * self.fx_ * reference_pv01_raw / 1e-4` (around line 1445). This formula assumes R's own "coupon" enters R's PV *additively*, with `d(R.value)/d(coupon_R) == +R.pv01()/1e-4` -- true for the coupon-style engines the base class was verified against in the session that added it (`ValuationEngineProductOvernightIndexSwap`'s fixed rate, `ValuationEngineProductCashDeposit`'s coupon, etc: `PV = ... + coupon*annuity`, `pv01() == d(PV)/d(coupon)*1e-4` by construction there).

For `ValuationEngineProductGenericForward` -- what this subclass's T and R *always* are -- the payoff is `sign*notional*tau*(F-K)*df`: `K` (`coupon`) enters *subtracted* from `F`, and `pv01()` is explicitly defined as `d(value)/dF` (`ValuationEngineProductGenericForward.pv01()`), not `d(value)/dK`. So here `d(R.value)/d(coupon_R) == -R.pv01()/1e-4`, the *opposite* sign from what the inherited formula assumes. Since `coupon_R = -spread_` (`d(coupon_R)/d(spread_) = -1`), the true `d(value_)/d(spread_)` for this class is `+sign*scale*fx*R.pv01()/1e-4` -- exactly the opposite sign of what `spread_pv01_unit_` (and therefore `pv01()` and `par_rate_or_spread()`, both built on it) actually compute.

Verified below against two independent, non-engine oracles: (1) a finite difference of `value_` from directly bumping `product.spread` and rebuilding the whole `ProductGenericForwardSpread` (unambiguous ground truth, does not touch any of this class's own formulas), and (2) the closed-form `+sign*scale*fx*R.pv01()/1e-4` derived above. Both agree with each other; the engine's own `pv01()` is their exact negation. `par_rate_or_spread()` inherits the same error: repricing at the engine's own "par" spread does *not* zero the PV, while repricing at the independently derived true par spread does, to float precision. Not patched, per this session's instructions -- flagged here and in the session report rather than fixed in `fixedincomelib/`.

In [79]:
true_dvalue_dspread = gfs_engine.sign_ * NOTIONAL * gfs_tau * float(gfs_df_pay.detach())
true_pv01 = true_dvalue_dspread * 1e-4
true_par_spread = float(gfs_F_T.detach()) - float(gfs_F_R.detach())


def gfs_value_at_spread(spread):
    p = ProductGenericForwardSpread(
        EFFECTIVE, GFS_TERM, PayOrReceive.RECEIVE, Currency('USD'), NOTIONAL, spread,
        basis_index=libor_3m, reference_index=sofr_index,
        accrual_basis=gfs_accrual_basis,
        business_day_convention=ib_biz_conv, holiday_convention=ib_hol_conv,
        payment_business_day_conv=ib_biz_conv, payment_holiday_conv=ib_hol_conv,
        compounding_method=CompoundingMethod.SIMPLE,
    )
    e = ValuationEngineProductGenericForwardSpread(yc, vpc, p, ValuationRequest.PV)
    e.calculate_value()
    return _to_float(e.value)


fd_pv01 = gfs_value_at_spread(GFS_SPREAD + 1e-4) - gfs_value_at_spread(GFS_SPREAD)
print('finite-diff pv01 (ground truth, bumps product.spread directly):', fd_pv01)
print('independent closed form (+sign*N*tau*df)                     :', true_pv01)
assert abs(fd_pv01 - true_pv01) < 1.0

engine_pv01 = gfs_engine.pv01()
print('engine pv01()  :', engine_pv01)
print('true pv01      :', true_pv01)
print('NOT ASSERTED EQUAL -- engine pv01() is the exact negation of the ground truth (see markdown note above).')

engine_par = gfs_engine.par_rate_or_spread()
print('engine par_rate_or_spread():', engine_par, ' true par spread (F_T - F_R):', true_par_spread)

pv_at_true_par = gfs_value_at_spread(true_par_spread)
print('PV at true par spread (should be ~0):', pv_at_true_par)
assert abs(pv_at_true_par) < 1e-2

pv_at_engine_par = gfs_value_at_spread(engine_par)
print('PV at engine par_rate_or_spread() (NOT ~0 -- consequence of the same sign error):', pv_at_engine_par)
print('NOT ASSERTED EQUAL TO ZERO -- see markdown note above; suspected bug, reported rather than patched.')


finite-diff pv01 (ground truth, bumps product.spread directly): 967.0765379429213
independent closed form (+sign*N*tau*df)                     : 967.0765379429106
engine pv01()  : -967.0765379429106
true pv01      : 967.0765379429106
NOT ASSERTED EQUAL -- engine pv01() is the exact negation of the ground truth (see markdown note above).
engine par_rate_or_spread(): -0.04560579990684913  true par spread (F_T - F_R): 0.04780579990684912
PV at true par spread (should be ~0): 0.0
PV at engine par_rate_or_spread() (NOT ~0 -- consequence of the same sign error): -903361.6655153994
NOT ASSERTED EQUAL TO ZERO -- see markdown note above; suspected bug, reported rather than patched.


### 14c. `create_cash_flows_report()` -- two rows, sums to `value_` exactly

In [80]:
gfs_cf = gfs_engine.create_cash_flows_report()
print('schema:', gfs_cf.schema)
print('rows  :', len(gfs_cf.content))
assert len(gfs_cf.content) == 2

pv_col = gfs_cf.schema.index('PV')
summed_cf = sum(row[pv_col] for row in gfs_cf.content)
print('sum of cf PV rows:', summed_cf, ' engine value_:', _to_float(gfs_engine.value))
assert abs(summed_cf - _to_float(gfs_engine.value)) < 1e-4
print('PASSED')


schema: ['PRODUCT_TYPE', 'VALUATION_ENGINE_TYPE', 'LEG_ID', 'CASHFLOW_ID', 'PAY_OR_RECEIVE', 'NOTIONAL', 'PAY_DATE', 'FORECASTED_AMOUNT', 'PV', 'DISCOUNG FACTOR', 'START_DATE', 'END_DATE', 'INDEX_OR_FIXED', 'INDEX_VALUE']
rows  : 2
sum of cf PV rows: -451680.8327576997  engine value_: -451680.8327576997
PASSED


### 14d. Suspected library bug: fully matured -> `ZeroDivisionError`

`fixedincomelib/yield_curve/valuation_engine.py`, `ValuationEngineProductGenericSpread.calculate_value`, the line `self.ratio_ = reference_pv01_raw / target_pv01_raw` (around line 1427), computed unconditionally. Once `value_date_` is past both legs' `payment_date_`, `ValuationEngineProductGenericForward.pv01()` (the closed form `sign*notional*tau*df*1e-4`) evaluates to exactly `0.0` for *both* legs, since `df_` itself is set to `0.0` in that engine's `value_date_ > payment_date_` branch -- so `calculate_value()` here divides `0.0/0.0` and raises `ZeroDivisionError`, rather than returning a matured `value_ == 0.0` the way every other settlement branch in this file (and this same product family's own atomic legs) does. Demonstrated below via a caught exception, not a hard assert of "correct" behavior -- not patched, per this session's instructions.

In [81]:
gfs_matured_date = add_period(gfs_product.pay_date, Period('1D'), ib_biz_conv, ib_hol_conv)
yc_gfs_matured = build_model(gfs_matured_date)
gfs_matured_engine = ValuationEngineProductGenericForwardSpread(yc_gfs_matured, vpc, gfs_product, ValuationRequest.PV)

matured_raised = None
try:
    gfs_matured_engine.calculate_value()
except ZeroDivisionError as e:
    matured_raised = e
print('fully matured calculate_value() raised:', type(matured_raised), matured_raised)
assert matured_raised is not None
print('CONFIRMED (not asserted as correct behavior) -- see markdown note above.')


fully matured calculate_value() raised: <class 'ZeroDivisionError'> float division by zero
CONFIRMED (not asserted as correct behavior) -- see markdown note above.


### 14e. `get_risk()` -- not supported (documented behavior, matches CLAUDE.md)

`value_` is combined from `_to_float(self.reference_engine_.value)`/`_to_float(self.target_engine_.value)` in the inherited `calculate_value` -- both sub-engines' live torch graphs are detached to plain floats *before* being combined, so `gfs_engine.value_` itself is a plain Python float, never a `torch.Tensor`. `ValuationEngineProduct.get_risk`'s default therefore skips `.backward()` entirely and just returns whatever's currently on the model's gradient accumulator (`model.get_gradient(reset=True)`) -- on a fresh model that's never had `.backward()` called on it, that's all zeros. This matches CLAUDE.md's documented limitation ("Curve risk for a generic spread is not currently supported") exactly -- tested below as the expected behavior, not as a shortcoming of this test.

In [82]:
yc_fresh = build_model(VALUE_DATE)
gfs_fresh_engine = ValuationEngineProductGenericForwardSpread(yc_fresh, vpc, gfs_product, ValuationRequest.PV)
gfs_fresh_engine.calculate_value()
assert not isinstance(gfs_fresh_engine.value_, torch.Tensor)

yc_fresh.get_gradient(reset=False)
grad = np.zeros(sum(yc_fresh.gradient_lengths_))
gfs_fresh_engine.get_risk(gradient=grad)
print('get_risk() on a fresh model (value_ is a plain float, not a live graph):', grad)
assert np.allclose(grad, 0.0)
print('PASSED (documents the not-supported limitation, per CLAUDE.md)')


get_risk() on a fresh model (value_ is a plain float, not a live graph):

 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
PASSED (documents the not-supported limitation, per CLAUDE.md)


## Summary

In [83]:
print('All tests passed:')
print(' - ValuationEngineProductFixedAccrued: PV, pv01, settlement branches, get_risk vs FD')
print(' - ValuationEngineProductOvernightIndexCompositeCashflow: PV (incl. leverage), pv01, settlement branches, get_risk vs FD')
print(' - ValuationEngineProductIBORIndexCashflow: PV, pv01, settlement branches, get_risk vs FD')
print(' - ValuationEngineProductIBORCompoundingCashflow: PV (both compounding methods), pv01, settlement branches, get_risk vs FD')
print(' - ValuationEngineProductInterestRateStream: PV at 3 value dates, par_rate_or_spread, pv01 vs FD, get_risk vs FD')
print(' - ValuationEngineProductOvernightIndexSwap: PV vs per-cashflow sum, par/pv01 vs FD, matured branch, get_risk vs FD')
print(' - ValuationEngineProductOvernightIndexBasisSwap: PV vs per-cashflow sum, par/pv01 vs FD, matured branch, get_risk vs FD')
print(' - ValuationEngineProductOISBasisSwap: PV vs two independent closed forms, par/pv01 vs FD, matured branch, get_risk vs FD')
print(' - ValuationEngineProductOvernightIndexFuture: PV, pv01/grad_at_par closed forms, MTM/settlement branches, get_risk vs FD')
print(' - ValuationEngineProductFRAOrFixing: PV/par/pv01 (true FRA), settlement branches, get_risk vs FD, overnight-index behavior documented;')
print('   two suspected library bugs in the early-settlement-factor branching flagged (not patched) -- see 10a-note and 10d-note')
print(' - ValuationEngineProductCashDeposit: PV (forward-starting and already-started), par/pv01 closed forms, get_risk vs FD')
print(' - ValuationEngineProductPortfolio (via ValuationEngineProductInterestRateStream): product_risk sums to get_risk, RiskValParam-driven get_risk_report() PORTFOLIO/PRODUCT levels, element order, no-accumulate contract')
print(' - ValuationEngineProductGenericForward: PV/pv01 (both SIMPLE and CONTINUOUS compounding), settlement branches, get_risk vs FD')
print(' - ValuationEngineProductGenericSpread: confirmed blocked (create_cash_flows_report commented out -> TypeError at construction);')
print('   shared calculate_value/par_rate_or_spread/pv01 logic exercised indirectly via ValuationEngineProductGenericForwardSpread instead')
print(' - ValuationEngineProductGenericForwardSpread: value_ vs independent closed form, create_cash_flows_report row sum;')
print('   two further suspected library bugs flagged (not patched) -- see 14b (pv01/par sign error) and 14d (matured -> ZeroDivisionError);')
print('   get_risk() not-supported behavior (plain-float value_) documented as expected, matching CLAUDE.md')


All tests passed:
 - ValuationEngineProductFixedAccrued: PV, pv01, settlement branches, get_risk vs FD
 - ValuationEngineProductOvernightIndexCompositeCashflow: PV (incl. leverage), pv01, settlement branches, get_risk vs FD
 - ValuationEngineProductIBORIndexCashflow: PV, pv01, settlement branches, get_risk vs FD
 - ValuationEngineProductIBORCompoundingCashflow: PV (both compounding methods), pv01, settlement branches, get_risk vs FD
 - ValuationEngineProductInterestRateStream: PV at 3 value dates, par_rate_or_spread, pv01 vs FD, get_risk vs FD
 - ValuationEngineProductOvernightIndexSwap: PV vs per-cashflow sum, par/pv01 vs FD, matured branch, get_risk vs FD
 - ValuationEngineProductOvernightIndexBasisSwap: PV vs per-cashflow sum, par/pv01 vs FD, matured branch, get_risk vs FD
 - ValuationEngineProductOISBasisSwap: PV vs two independent closed forms, par/pv01 vs FD, matured branch, get_risk vs FD
 - ValuationEngineProductOvernightIndexFuture: PV, pv01/grad_at_par closed forms, MTM/settl